# SHAP feature-selected model

In [ ]:
# --- standard library ---
import json
import os
import re
import warnings
from collections import Counter
from fractions import Fraction

# --- numeric / data ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.ndimage import gaussian_filter1d
from scipy.stats import beta

# --- plotting ---
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import pyplot, cbook
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator

# --- modelling ---
import xgboost as xgb
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    accuracy_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    GroupShuffleSplit,
    GroupKFold,
    GridSearchCV,
)
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from xgboost import XGBClassifier

# --- notebook ---
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Image, display

InteractiveShell.ast_node_interactivity = "all"
pd.set_option('display.max_columns', None)

# Create a truncated version of the Blues colormap
colourmap = mcolors.LinearSegmentedColormap.from_list(
    'Blues_truncated',
    plt.cm.Blues(np.linspace(0.15, 1.0, 256))
)


In [172]:
SEED = 43
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)

In [ ]:
# different data for different imputation methods
dfnames = ['interpolation', 'uncertainty', 'NOimputation']
i = 2
DATA_SOURCE = dfnames[i]
PLOT_PATH = f'/path/to/plots/{DATA_SOURCE}/'
DATA_PATH = f"/path/to/data/{DATA_SOURCE}/"
PLOT_DATA_PATH = f'/path/to/plots/{DATA_SOURCE}/plot_data/'
os.makedirs(PLOT_DATA_PATH, exist_ok=True)
print("Data source:", DATA_SOURCE)

## Data

In [175]:
# save test train splits for later use
top_features = pd.read_csv(DATA_PATH + "SHAP_selected_features_over_0.01.csv")['feature'].tolist()

y_tr = pd.read_csv(DATA_PATH + "y_train.csv")['infektion_binary']
y_test = pd.read_csv(DATA_PATH + "y_test.csv")['infektion_binary']
group_tr = pd.read_csv(DATA_PATH + "group_train.csv")['henkilotunnus']
group_test = pd.read_csv(DATA_PATH + "group_test.csv")['henkilotunnus']
top_features = pd.read_csv(DATA_PATH + "SHAP_selected_features_over_0.01.csv")['feature'].tolist()
df_test = pd.read_csv(DATA_PATH + "df_test.csv")

X_tr = pd.read_csv(DATA_PATH + "X_train.csv")[top_features]
X_test = pd.read_csv(DATA_PATH + "X_test.csv")[top_features]

X_tr_all = pd.read_csv(DATA_PATH + "X_train.csv").drop(columns='Unnamed: 0')
X_test_all = pd.read_csv(DATA_PATH + "X_test.csv").drop(columns='Unnamed: 0')
metrics_df = pd.read_csv(PLOT_PATH+'metrics_table.csv').drop(columns='Unnamed: 0')

df_tr = pd.read_csv(DATA_PATH + "df_tr.csv").drop(columns='Unnamed: 0')

## Functions

In [177]:
def proba_to_labels(p, threshold: float):
    """Convert probabilities to 0/1 labels."""
    return (p >= threshold).astype(int)

In [178]:

def plot_cv_mean_confusion_matrix_rowpct(
    mean_mat, std_mat=None,
    *,
    mean_cnt=None, std_cnt=None,     # <-- NEW (optional raw counts)
    title="CV mean (± std) confusion matrix (row %)",
    threshold=None,
    save_path=None,
    cmap="viridis",
    text_color=None,
    figsize=(6, 5),
    alpha=1.0,
):
    """
    Plot a single 2x2 macro-averaged confusion matrix where `mean_mat` and `std_mat`
    are row-normalized FRACTIONS (0..1). This renders as percentages.

    Optionally, also display raw-count mean (`mean_cnt`) and std (`std_cnt`) per cell.
    Shapes expected: (2, 2) for all matrices.
    """
    mean_mat = np.asarray(mean_mat, float)
    std_mat  = None if std_mat is None else np.asarray(std_mat, float)

    # Percent view for heatmap
    perc = mean_mat * 100.0
    stdp = None if std_mat is None else (std_mat * 100.0)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(perc, cmap=cmap, alpha=alpha, vmin=0, vmax=100)

    # ---- Annotations in each cell ----
    for i in range(perc.shape[0]):
        for j in range(perc.shape[1]):
            # First line: percent ± std%
            if stdp is None:
                line1 = f"{perc[i, j]:.1f}%"
            else:
                line1 = f"{perc[i, j]:.1f}% ± {stdp[i, j]:.1f}"

            # Second line: count ± std (if provided)
            line2 = None
            if mean_cnt is not None:
                mcnt = float(mean_cnt[i, j])
                if std_cnt is not None:
                    scnt = float(std_cnt[i, j])
                    line2 = f"{mcnt:.1f} ± {scnt:.1f}"
                else:
                    line2 = f"{mcnt:.1f}"

            txt = line1 if line2 is None else f"{line1}\n{line2}"

            # text color: auto-contrast unless overridden
            color = "black" if (text_color is None and perc[i, j] <= perc.max()/2) else \
                    ("white" if text_color is None else text_color)

            ax.text(j, i, txt, ha="center", va="center", fontsize=13, color=color)

    # Axes labels/ticks
    # axis labels font size
    ax.set_xlabel("Predicted label", fontsize=13)
    ax.set_ylabel("True label", fontsize=13)

    # tick label font size
    ax.tick_params(axis="both", which="major", labelsize=13)

    # if you set custom tick labels:
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels([0, 1], fontsize=13)
    ax.set_yticklabels([0, 1], fontsize=13)

    full_title = title
    ax.set_title(full_title, fontsize=15)

    # Colorbar in % scale
    cbar = plt.colorbar(im)
    cbar.set_label("% (row-normalized)", rotation=90, fontsize=13)

    cbar.ax.tick_params(labelsize=13)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")



In [179]:
def calc_pnv(y_true, y_pred):
    """
    Compute PNV (Negative Predictive Value) for binary data.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Predicted negative = 0
    TN = np.sum((y_pred == 0) & (y_true == 0))  # true negatives
    FN = np.sum((y_pred == 0) & (y_true == 1))  # false negatives

    if TN + FN == 0:
        return np.nan  # no predicted negatives → PNV undefined

    return TN / (TN + FN)

In [180]:
def plot_confusion_matrix_with_percentages(
    y_true,
    y_pred,
    labels=None,
    percent_type="row",   # 'row', 'column', or 'all'
    title=None,
    cmap="viridis",
    threshold=None,
    save_path=None,
    figsize=None,
    text_color=None,
    plot_data=False,          # <-- NEW
):
    from sklearn.metrics import confusion_matrix
    import numpy as np
    import matplotlib.pyplot as plt

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if labels is None:
        labels = np.unique(y_true)

    # Calculate percentages
    if percent_type == "row":
        sums = cm.sum(axis=1, keepdims=True)
        perc = cm / np.where(sums == 0, 1, sums) * 100
        perc_label = "Row %"
    elif percent_type == "column":
        sums = cm.sum(axis=0, keepdims=True)
        perc = cm / np.where(sums == 0, 1, sums) * 100
        perc_label = "Col %"
    elif percent_type == "all":
        total = cm.sum()
        perc = (cm / total * 100) if total != 0 else np.zeros_like(cm, dtype=float)
        perc_label = "All %"
    else:
        raise ValueError("percent_type must be 'row', 'column', or 'all'")

    # Annotation with count and percentage
    annot = np.empty_like(cm).astype(str)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]}\n({perc[i, j]:.0f}%)"

    fig, ax = plt.subplots(figsize=figsize)

    # ✅ Heatmap uses PERCENTAGES so colors + colorbar are %
    im = ax.imshow(perc, cmap=cmap, vmin=0, vmax=100)
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=13)
    cbar.set_label("Percent (%)", fontsize=13)

    # Show numbers
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if text_color is None:
                # background is % (0..100), so use 50 as midpoint
                cell_color = "black" if perc[i, j] > 50 else "white"
            else:
                cell_color = text_color

            ax.text(
                j, i, annot[i, j],
                ha="center", va="center",
                fontsize=13,
                color=cell_color
            )

    ax.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted label",
        ylabel="True label",
    )

    ax.set_xlabel("Predicted label", fontsize=14)
    ax.set_ylabel("True label", fontsize=14)
    ax.tick_params(axis="x", labelsize=13)
    ax.tick_params(axis="y", labelsize=13)

    full_title = title if title else f"Confusion Matrix with {perc_label}"
    ax.set_title(full_title, fontsize=14)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    # ---- source data for the journal --------------------------------------
    if plot_data:
        _rows = []
        for i, t in enumerate(labels):
            for j, p in enumerate(labels):
                _rows.append({
                    "true_label": t,
                    "predicted_label": p,
                    "count": int(cm[i, j]),
                    "percent": float(perc[i, j]),
                    "percent_basis": percent_type,
                })
        _name = save_path if save_path is not None else "".join(
            c if (c.isalnum() or c in "-_") else "_" for c in full_title)
        _note = f"percent_type={percent_type}"
        if threshold is not None:
            _note += f"; threshold={threshold}"
        write_plot_data(_name, pd.DataFrame(_rows),
                        out_dir=PLOT_DATA_PATH, note=_note)
        
    plt.show()


In [181]:
def custom_score(y_true, y_pred, group_ids, threshold, eval=False):
    # Binary prediction
    y_bin = (y_pred >= threshold).astype(int)
    print(f"f1 before mod: {f1_score(y_true, y_bin)}")
    # Create working DataFrame
    df = pd.DataFrame({
        'henkilotunnus': group_ids,
        'true_label': y_true,
        'predicted_label': y_bin
    })

    df['misclassified'] = (df['true_label'] != df['predicted_label']).astype(int)
    df['misclassified_proximity'] = 0

    # Set proximity rule
    for name, group in df.groupby('henkilotunnus'):
        for i in range(len(group) - 1):
            row = group.iloc[i]
            next_row = group.iloc[i + 1]

            if (
                row['misclassified'] == 1
                and row['true_label'] == 0
                and row['predicted_label'] == 1
                and next_row['true_label'] == 1
            ):
                df.loc[group.index[i], 'misclassified_proximity'] = 1
 
    # Adjust predictions
    adjusted_pred = df['predicted_label'].copy()
    mask = (df['misclassified'] == 1) & (df['misclassified_proximity'] == 1)
    adjusted_pred[mask] = df['true_label'][mask]

    if eval:
        return df, f1_score(df['true_label'], adjusted_pred), adjusted_pred
    else:
        return f1_score(df['true_label'], adjusted_pred)

## Feature selection

In [182]:
neg_cnt = y_tr.value_counts()[0]
pos_cnt = y_tr.value_counts()[1]
scale1 = (neg_cnt / pos_cnt) ** 0.5
scale2 = (neg_cnt / pos_cnt)

In [183]:
# Same hyperparams as XGB with all variables --> selected using hyperparam search later in the code
preset = {
    'colsample_bytree': 0.7,
    'learning_rate': 0.05,
    'max_depth': 6,
    'n_estimators': 100,
    'scale_pos_weight': scale1,
    'subsample': 0.8,
    "eval_metric":'logloss',
    "use_label_encoder":False,
    "verbosity":0,
    "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
    "early_stopping_rounds":10,
    #"random_state":SEED
    }

### K-fold F1 with top features added one at a time
- Features added one at a time
- For each added top feature this is repeated 5 times in k-fold; the mean and std of the results are reported

In [184]:
MULTI=3

In [185]:
def custom_f1(y_true, y_pred, multi=3):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fp_adj = fp / multi
    precision = tp / (tp + fp_adj) if (tp + fp_adj) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

In [186]:
featurenames = {
  "temperature"                       : "Body temperature (°C)",
  "b_neut"                            : "ANC (10⁹/L)",
  "b_leuk"                            : "WBC (10⁹/L)",
  "temperature_trend_from_2_d"        : "Temperature \n2-day trend (°C)",
  'b_leuk_trend_from_7_d'             : "WBC 7-day trend (10⁹/L)",
  'p_crp_trend_from_3_d'              : "1–3-day CRP trend (mg/L)",
  'sykli_IND'                         : "Induction phase",
  "bneut_accumulated_max_050_cycle"   : "Duration of ANC\n(≤0.5 10⁹/L) per cycle (days)",
  "bneut_accumulated_max_050_total"   : "Duration of ANC\n(≤0.5 10⁹/L) per regimen (days)",
  "bneut_accumulated_max_005_total"   : "Duration of ANC\n(≤0.05 10⁹/L) per regimen (days)",
  "countdown"                         : "Time since \ncycle start (days)",
  "cycle_number"                      : "Cycle number",

  "e_mcv_mean_last_values_7_days"     : "4–7-day mean MCV (fL)",
  "p_tt_mean_last_values_7_days"      : "4–7-day tromboplastin time trend (%)",
  "p_crp_trend_from_7_d"              : "4–7-day CRP trend (mg/L)",
  "p_afos_mean_last_values_7_days"    : "ALP mean 7-day trend (U/L)",
  "p_k_mean_last_values_7_days"       : "K mean 7-day trend (mmol/L)",
  "bm_blast_p"                        : "BM blast (%)",
  "b_trom"                            : "PLT (10⁹/L)",
  "b_trom_trend_from_7_d"             : "4–7-day PLT trend (10⁹/L)",
  "p_tt_mean_last_values_3_days"      : "Tromboplastin time mean 3-day trend (%)",
  "b_trom_mean_last_values_7_days"    : "4–7-day PLT trend (10⁹/L)",
  "b_neut_mean_last_values_7_days"    : "4–7-day ANC trend (10⁹/L)",
  "b_eryt_mean_last_values_3_days"    : "RBC mean 3-day trend (10$^12$/L)",
  "b_ly_trend_from_7_d"               : "Lymphocytes 7-day trend (10⁹/L)",

  "ab_max_10d"                        : 'Ab initiated within the last 10 days',

}

In [187]:
# ---------------------------------------------------------------------------
# Source-data export for figures (journal request)
# ---------------------------------------------------------------------------

_IMG_EXT = {".png", ".pdf", ".svg", ".eps", ".jpg", ".jpeg", ".tif", ".tiff"}


def _plot_data_stem(name):
    base = os.path.basename(str(name).rstrip("/"))
    root, ext = os.path.splitext(base)
    return root if ext.lower() in _IMG_EXT else base


def write_plot_data(name, tables, out_dir=None, note="", manifest=True,
                    verbose=True):
    """One table -> <PLOT_DATA_PATH>/<stem>.csv
       Several    -> <PLOT_DATA_PATH>/<stem>/<stem>__<part>.csv"""
    out_dir = out_dir or PLOT_DATA_PATH
    stem = _plot_data_stem(name)
    if isinstance(tables, pd.DataFrame):
        tables = {"data": tables}
    tables = {k: df for k, df in tables.items() if df is not None and len(df)}
    if not tables:
        if verbose:
            print(f"[plot_data] {stem}: nothing to write")
        return []

    target = out_dir if len(tables) == 1 else os.path.join(out_dir, stem)
    os.makedirs(target, exist_ok=True)

    paths, rows = [], []
    for part, df in tables.items():
        fname = f"{stem}.csv" if len(tables) == 1 else f"{stem}__{part}.csv"
        path = os.path.join(target, fname)
        df.to_csv(path, index=False)
        paths.append(path)
        rows.append({"figure": stem,
                     "part": "" if len(tables) == 1 else part,
                     "file": os.path.relpath(path, out_dir),
                     "n_rows": len(df),
                     "columns": "; ".join(map(str, df.columns)),
                     "note": note})

    if manifest:
        idx = os.path.join(out_dir, "plot_data_index.csv")
        new = pd.DataFrame(rows)
        if os.path.exists(idx):
            old = pd.read_csv(idx, keep_default_na=False)
            old = old[old["figure"].astype(str) != stem]   # rerun replaces
            new = pd.concat([old, new], ignore_index=True)
        new.to_csv(idx, index=False)

    if verbose:
        print(f"[plot_data] {stem}: " +
              ", ".join(os.path.relpath(p, out_dir) for p in paths))
    return paths

In [188]:
%%capture 

# initial threshold taken from model using all features
INIT_THRESHOLD = 0.20
one_kfold = False

if one_kfold:
    from sklearn.model_selection import KFold  # or GroupKFold if you have groups
    import numpy as np
    import matplotlib.pyplot as plt

    # Prepare storage
    train_f1_means = []
    train_f1_stds = []
    val_f1_means = []
    val_f1_stds = []
    feature_names_added = []

    kf = StratifiedGroupKFold(n_splits=5)

    for k in range(1, len(top_features[:16]) + 1):
        selected = top_features[:k]
        feature_names_added.append(selected[-1])  # Or append a string of all features if you prefer

        fold_train_f1 = []
        fold_val_f1 = []

        for train_idx, val_idx in kf.split(X_tr, y_tr, groups=group_tr):
            X_train, X_val = X_tr.iloc[train_idx][selected], X_tr.iloc[val_idx][selected]
            y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]

            model = XGBClassifier(**preset, seed=SEED-1, random_state=SEED-1);
            model.fit(
                X_train.values, y_train.values,
                eval_set=[(X_val.values, y_val.values)],
                verbose=0
            );

            if INIT_THRESHOLD is not None:
                y_pred_train = model.predict_proba(X_train.values)[:, 1]
                y_train_pred = (y_pred_train >= INIT_THRESHOLD).astype(int)

                y_pred_val = model.predict_proba(X_val.values)[:, 1]
                y_val_pred = (y_pred_val >= INIT_THRESHOLD).astype(int)

            else:
                y_train_pred = model.predict(X_train.values); # TODO threshold?
                y_val_pred = model.predict(X_val.values); # TODO threshold?

            fold_train_f1.append(custom_f1(y_train, y_train_pred))
            fold_val_f1.append(custom_f1(y_val, y_val_pred))

        # Store mean/std for this feature set
        train_f1_means.append(np.mean(fold_train_f1))
        train_f1_stds.append(np.std(fold_train_f1))
        val_f1_means.append(np.mean(fold_val_f1))
        val_f1_stds.append(np.std(fold_val_f1))

    print("max-min values of mean f1 (first 15):", max(val_f1_means[:15]), min(val_f1_means[:15]))

    # Plot

    #tab_blue = mcolors.to_rgb("tab:blue")
    tab_blue = "#C32B26"
    lighter_blue = "#FD7B79"
    plt.figure(figsize=(11, 4 + 0.27*len(feature_names_added[:15])))

    plt.errorbar(
        range(1, len(train_f1_means[:15]) + 1), train_f1_means[:15], yerr=train_f1_stds[:15],
        fmt='-o', capsize=4, label='Training', color=tab_blue
    )
    plt.errorbar(
        range(1, len(val_f1_means[:15]) + 1), val_f1_means[:15], yerr=val_f1_stds[:15],
        fmt='-s', capsize=4, label='Validation', color=lighter_blue
    )
    plt.xlabel('Features', fontsize=14)
    plt.ylabel('Custom F1 score', fontsize=14)
    plt.ylim(0.2, 1)  # Set y-limits here!
    plt.yticks(fontsize=13)
    
    labels = [featurenames.get(f, f) for f in feature_names_added[:15]]
    plt.xticks(
        range(1, len(labels) + 1),
        labels,
        rotation=45,
        ha="right",
        fontsize=13
    )
    plt.legend(fontsize=13)
    plt.grid(True, alpha=0.5)
    plt.tight_layout()
    plt.savefig(PLOT_PATH+'simple_model/incremental_f1_scores_by_SHAP_features_GroupKFold.png', dpi=600, bbox_inches='tight')

        # ---- source data for the journal --------------------------------------
    FIG_NAME = 'incremental_f1_scores_by_SHAP_features_GroupKFold'

    _tm = np.asarray(train_f1_means[:15], float)
    _ts = np.asarray(train_f1_stds[:15],  float)
    _vm = np.asarray(val_f1_means[:15],   float)
    _vs = np.asarray(val_f1_stds[:15],    float)
    _n  = len(_tm)

    write_plot_data(
        FIG_NAME,
        pd.DataFrame({
            "step":                np.arange(1, _n + 1),
            "feature_added":       list(feature_names_added[:15]),
            "feature_label":       [re.sub(r"\s*\n\s*", " ", str(l)) for l in labels],
            "train_f1_mean":       _tm,
            "train_f1_sd":         _ts,
            "train_whisker_lower": _tm - _ts,
            "train_whisker_upper": _tm + _ts,
            "val_f1_mean":         _vm,
            "val_f1_sd":           _vs,
            "val_whisker_lower":   _vm - _vs,
            "val_whisker_upper":   _vm + _vs,
        }),
        note=(f"points = mean custom F1 over {getattr(kf, 'n_splits', '?')} "
              f"{type(kf).__name__} folds; whiskers = SD (numpy ddof=0); "
              f"decision threshold={INIT_THRESHOLD}; "
              f"features added cumulatively in SHAP rank order; "
              f"{_n} of {len(train_f1_means)} computed steps plotted"),
    )
    
    plt.show();


In [ ]:
Image(filename=PLOT_PATH+'simple_model/incremental_f1_scores_by_SHAP_features_GroupKFold.png', width=1000, height=1000) 

### Select n first columns and retrain

In [190]:
old_cols = ['countdown', 'b_neut', 'b_leuk', 'sykli_IND', 'temperature', 'p_crp_trend_from_7_d', 'ab_max_10d']
monotone_const = (0, -1, -1, 1, 1, 1, 0)


In [ ]:

# manually checked best number of features for each imputation method
features_dict = {'interpolation':9, 'uncertainty':13, 'NOimputation':7}

keep = features_dict[DATA_SOURCE]
best_model = XGBClassifier(seed=SEED, random_state=SEED)

# select columns
X_tr_drop = X_tr.iloc[:, : keep]
X_test_drop = X_test.iloc[:, : keep]
X_test_drop.to_csv(DATA_PATH + "X_test_drop.csv")

print('Number of features:', len(X_tr_drop.columns))
X_tr_drop.columns

In [ ]:
# lets select the same columns but i want them to match the order in monotone_constraints...
# select columns
X_tr_drop = X_tr[old_cols]
X_test_drop = X_test[old_cols]
X_test_drop.to_csv(DATA_PATH + "X_test_drop.csv")

print('Number of features:', len(X_tr_drop.columns))
X_tr_drop.columns

## Hyperparameter tuning

In [193]:
GS = False
if GS:

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    for train_idx, val_idx in splitter.split(X_tr_drop, y_tr, groups=group_tr):
        X_train, X_val = X_tr_drop.iloc[train_idx], X_tr_drop.iloc[val_idx]
        y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]
        group_train, group_val = group_tr.iloc[train_idx], group_tr.iloc[val_idx]

    scale1 = (y_train.value_counts()[0]/y_train.value_counts()[1])**1/2
    scale2 = (y_train.value_counts()[0]/y_train.value_counts()[1])

    # Your preset parameters
    preset = {
        "eta": 1,
        "max_depth": 5,
        "n_estimators": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "learning_rate": 0.1,
        "eval_metric": 'logloss',
        "use_label_encoder": False,
        "verbosity": 0,
        "objective": 'binary:logistic'
    }

    # Define your parameter grid for grid search
    param_grid = {
        "max_depth": [4, 5, 6, 8, 9],
        "n_estimators": [70, 100, 115, 130],
        "learning_rate": [0.05, 0.07, 0.1],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "scale_pos_weight":[1, scale1, scale1*2, scale2]

        # You can add/remove parameters as needed
    }

    # Instantiate the model (set static params from preset, don't override by grid search)
    xgb = XGBClassifier(
        eta=preset["eta"],
        eval_metric=preset["eval_metric"],
        use_label_encoder=preset["use_label_encoder"],
        verbosity=preset["verbosity"],
        objective=preset["objective"],
        seed=SEED,
        random_state=SEED,
    )

    # Set up GridSearchCV (assuming you have X_train, y_train)
    grid_search = GridSearchCV(
        estimator=xgb,
        param_grid=param_grid,
        scoring='neg_log_loss',  # Or 'accuracy', 'roc_auc', etc.
        cv=3,
        verbose=1,
        n_jobs=-1
    )

    # Fit the grid search (add early stopping if needed)
    grid_search.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    print("Best parameters found:", grid_search.best_params_)
    print("Best score:", grid_search.best_score_)


In [194]:
MULTI = 3

In [195]:
def custom_cost_3fp_fn(y_true, y_pred, c_fp=3.0, c_fn=1.0):
    """Cost = c_fp * FP + c_fn * FN (defaults: 3*FP + 1*FN)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    # return c_fp * fp + c_fn * fn
    return fp/c_fp + c_fn * fn

def custom_f1(y_true, y_pred, multi=3):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fp_adj = fp / multi
    precision = tp / (tp + fp_adj) if (tp + fp_adj) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0


In [196]:
n_folds = 5
thresholds = np.linspace(0, 1, 100)

In [197]:
def _fp_fn_for_preds(y_true, y_pred):
    _, fp, fn, _ = confusion_matrix(y_true, y_pred).ravel()
    return fp, fn

def threshold_metrics(y_true, y_prob, thresholds=np.linspace(0, 1, 100), c_fp=3.0, c_fn=1.0):
    f1s, custom_f1s, custom_f13, precisions, recalls, costs = [], [], [], [], [], []
    fps, fns = [], []
    for thresh in thresholds:
        y_pred = (y_prob > thresh).astype(int)

        f1s.append(f1_score(y_true, y_pred, zero_division=0))
        custom_f1s.append(custom_f1(y_true, y_pred))            # your existing
        custom_f13.append(custom_f1(y_true, y_pred, multi=MULTI))   # your existing
        precisions.append(precision_score(y_true, y_pred, zero_division=0))
        recalls.append(recall_score(y_true, y_pred, zero_division=0))

        fp, fn = _fp_fn_for_preds(y_true, y_pred)
        fps.append(fp); fns.append(fn)
        costs.append(3.0 * fp + 1.0 * fn)  # or c_fp/c_fn

    return (np.array(f1s), np.array(custom_f1s), np.array(custom_f13),
            np.array(precisions), np.array(recalls),
            np.array(costs), np.array(fps), np.array(fns))


### N estimators

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
for train_idx, val_idx in splitter.split(X_tr_drop, y_tr, groups=group_tr):

    X_train, X_val = X_tr_drop.iloc[train_idx], X_tr_drop.iloc[val_idx]
    y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]
    group_train, group_val = group_tr.iloc[train_idx], group_tr.iloc[val_idx]
 
# Make sure eval_metric is set; set n_estimators high and rely on early stopping
if i == 0:
    preset = {
        'colsample_bytree': 0.7,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': 2000,
        #'scale_pos_weigth': 1,
        'scale_pos_weight': 1,
        'subsample': 0.8,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }
elif i==1:
    preset = {
        'colsample_bytree': 0.8,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': 2000,
        'scale_pos_weight': 1,
        'subsample': 0.7,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }
else:
    preset = {
        'colsample_bytree': 0.7,
        'learning_rate': 0.07,
        'max_depth': 4,
        'n_estimators': 2000,
        'scale_pos_weight': 1,
        'subsample': 0.7,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        } 

model = XGBClassifier(**preset, seed=SEED, random_state=SEED)

model.fit(
    X_train.values, y_train.values,
    eval_set=[(X_train.values, y_train.values),
              (X_val.values,   y_val.values)],
    verbose=False
)

# --- Plot learning curves ---
results = model.evals_result()
train_loss = results['validation_0']['logloss']   # first tuple in eval_set
val_loss   = results['validation_1']['logloss']   # second tuple in eval_set
epochs = range(1, len(train_loss) + 1)

plt.figure()
plt.plot(epochs, train_loss, label='train logloss')
plt.plot(epochs, val_loss,   label='val logloss')
plt.axvline(model.best_iteration+1, linestyle='--',
            label=f'best iteration = {model.best_iteration+1}')
plt.xlabel('Number of trees')
plt.ylabel('Logloss')
plt.title('XGBoost learning curves')
plt.legend()
plt.show()
plt.savefig(PLOT_PATH+'simple_model/n_estimates_simple_model.png', dpi=600)

print("Best iteration:", model.best_iteration+1)
best_n_estimates = model.best_iteration+1


In [199]:
if i==0:
    preset = {
        'colsample_bytree': 0.7,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': min(best_n_estimates, 100),
        'scale_pos_weight': 1,
        'subsample': 0.8,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }
elif i==1:
    preset = {
        'colsample_bytree': 0.8,
        'learning_rate': 0.05,
        'max_depth': 4,
        'n_estimators': min(best_n_estimates, 100),
        'scale_pos_weight': 1,
        'subsample': 0.7,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }
else:
    preset = {
        'colsample_bytree': 0.7,
        'learning_rate': 0.07,
        'max_depth': 4,
        'n_estimators': min(best_n_estimates, 100),
        'scale_pos_weight': 1,
        'subsample': 0.7,
        "eval_metric":'logloss',
        "use_label_encoder":False,
        "verbosity":0,
        "objective":'binary:logistic', # pitäsköhä tähä tehä joku uus? 
        "early_stopping_rounds":10,
        }



## Model optimization
- Threshold tuning

### Threshold tuning

In [200]:
OVERSAMPLE = False
USE_PRESET = True
MONOTONE = True

In [201]:
def threshold_over_sampling(
        X_tr, y_tr, group_tr,
        multi=MULTI,
        n_folds=5,
        ratio='auto',
        random_state=SEED
        ):
    thresholds = np.linspace(0, 1, 100)

    all_f1s, all_custom_f1s, all_F1_3s = [], [], []
    all_precisions, all_recalls, all_costs, all_fps, all_fns = [], [], [], [], []

    #gkf = StratifiedGroupKFold(n_splits=n_folds)
    gkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=random_state)


    for train_idx, val_idx in gkf.split(X_tr, y_tr, groups=group_tr):  # your group var
        X_train, X_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]

        if OVERSAMPLE:    
            # --- NEW: oversample train only ---
            ros = RandomOverSampler(
                sampling_strategy=ratio,   # or e.g. 0.3, {1: 5000}, ...
                random_state=SEED,
            )
            X_train_res, y_train_res = ros.fit_resample(X_train, y_train)
        else:
            X_train_res = X_train
            y_train_res = y_train

        # --- XGBoost on oversampled data ---
        if USE_PRESET and MONOTONE:
            model = XGBClassifier(**preset, seed=SEED, random_state=SEED, monotone_constraints=monotone_const)
            
        elif USE_PRESET:
            model = XGBClassifier(**preset, seed=SEED, random_state=SEED)
        else:
            model = XGBClassifier(early_stopping_rounds=10, seed=SEED, random_state=SEED)

        model.fit(
            X_train_res.values, y_train_res.values,
            eval_set=[(X_val.values, y_val.values)],   # val set is NOT oversampled
            verbose=False,
        )

        # --- same threshold tuning as before ---
        y_prob = model.predict_proba(X_val.values)[:, 1]

        f1s, custom_f1s, F1_3s, precisions, recalls, costs, fps, fns = \
            threshold_metrics(y_val.values, y_prob, thresholds)

        all_f1s.append(f1s)
        all_F1_3s.append(F1_3s)
        all_precisions.append(precisions)
        all_recalls.append(recalls)
        all_costs.append(costs)
        all_fps.append(fps)
        all_fns.append(fns)

    # then your aggregation code:
    all_f1s        = np.array(all_f1s)
    all_F1_3s      = np.array(all_F1_3s)
    all_precisions = np.array(all_precisions)
    all_recalls    = np.array(all_recalls)
    all_costs      = np.array(all_costs)
    all_fps        = np.array(all_fps)
    all_fns        = np.array(all_fns)

    mean_f1        = all_f1s.mean(axis=0)
    mean_F1_3      = all_F1_3s.mean(axis=0)
    mean_precision = all_precisions.mean(axis=0)
    mean_recall    = all_recalls.mean(axis=0)
    mean_costs     = all_costs.mean(axis=0)
    mean_fp        = all_fps.mean(axis=0)
    mean_fn        = all_fns.mean(axis=0)
    
    # ... your feasible-mask + plotting code exactly as in screenshot
    feasible = mean_fp <= multi * mean_fn
    print(feasible)

    if feasible.any():
        # select lower limit
        nz = np.flatnonzero(~feasible)
        idx_best_constrained = int(nz[-1])
        print(idx_best_constrained)

    else:
        # If nothing feasible, fall back to the most conservative threshold (highest thr)
        idx_best_constrained = 0.5

    thresh_best_constrained = thresholds[idx_best_constrained]

    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Left axis: scores in [0,1]
    ax1.plot(thresholds, mean_f1,        label='F1')
    ax1.plot(thresholds, mean_F1_3,      label='Custom F1 (FP/3)')
    ax1.plot(thresholds, mean_precision, label='Precision')
    ax1.plot(thresholds, mean_recall,    label='Recall')
    ax1.set_xlabel('Threshold'); ax1.set_ylabel('Score'); ax1.set_ylim(0, 1); ax1.grid(True)

    # Shade infeasible region where FP > 3·FN
    ax1.fill_between(thresholds, 0, 1, where=~feasible, alpha=0.12, transform=ax1.get_xaxis_transform(),
                    label=f'Infeasible (FP > {multi}·FN)')

    # Right axis: cost curve (optional)
    ax2 = ax1.twinx()
    ax2.plot(thresholds, mean_costs, linestyle='--', label=f'Cost ({multi}·FP + FN)')
    ax2.set_ylabel(f'Cost ({multi}·FP + FN)')

    # Mark constrained-best threshold
    for ax in (ax1, ax2):
        ax.axvline(thresh_best_constrained, color='crimson', linestyle='--', linewidth=1)

    # Legends merged
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
    ax1.set_ylim(0, 1)
    ax1.set_yticks(np.arange(0, 1.01, 0.1))
    ax1.grid(True)

    # Simple annotations
    def annotate_with_arrow(ax, x, y, text, dx=0.02, dy=0.02, fmt="{:.3f}"):
        ax.annotate(
            f"{text}\n{fmt.format(x)}",
            xy=(x, y),
            xytext=(x + dx, y + dy),
            arrowprops=dict(arrowstyle='->'),
            bbox=dict(boxstyle='round', fc='w'),
        )

    annotate_with_arrow(ax2, thresh_best_constrained,  idx_best_constrained, 'Constraint best')             
    plt.tight_layout(); plt.show()
    plt.savefig(PLOT_PATH+'simple_model/thresholds_simple_model.png', dpi=600)

    return thresh_best_constrained


In [ ]:
if OVERSAMPLE:
    ratios = [0.2, 0.3, 0.4]
else:
    ratios = [None]

trs = []
for r in ratios:
    tr = threshold_over_sampling(
        X_tr_drop, y_tr, group_tr,
        multi=MULTI,
        #multi=1,
        n_folds=6,
        ratio=r,
        random_state=22
        )
    trs.append(tr)
    
THRESHOLD = np.round(trs, 1)
THRESHOLD

## Actual model training

Whole data training before evaluation.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)

for train_idx, val_idx in splitter.split(X_tr_drop, y_tr, groups=group_tr):
    X_train, X_val = X_tr_drop.iloc[train_idx], X_tr_drop.iloc[val_idx]
    y_train, y_val = y_tr.iloc[train_idx], y_tr.iloc[val_idx]
    group_train, group_val = group_tr.iloc[train_idx], group_tr.iloc[val_idx]

    if OVERSAMPLE:
    # ----- NEW: oversample training set only -----
        ros = RandomOverSampler(
            sampling_strategy="auto",   # CV ratio sweep above is disabled
            random_state=SEED,
        )
        X_train_res, y_train_res = ros.fit_resample(X_train, y_train)
    else:
        X_train_res = X_train
        y_train_res = y_train
    print(
        f"X_train: {X_train_res.shape}, y_train: {y_train_res.shape}, "
        f"X_val: {X_val.shape}, y_val: {y_val.shape}"
    )

    if USE_PRESET and MONOTONE:
        print('monotone')
        best_model = XGBClassifier(**preset, seed=SEED, random_state=SEED, monotone_constraints=monotone_const)
    elif USE_PRESET:
        best_model = XGBClassifier(**preset, seed=SEED, random_state=SEED)
    else:
        best_model = XGBClassifier(early_stopping_rounds=10, random_state=SEED)

    best_model.fit(
        X_train_res.values,
        y_train_res.values,
        eval_set=[(X_train_res.values, y_train_res.values),
                  (X_val.values,       y_val.values)],
        verbose=False,
    )

y_pred  = best_model.predict(X_val.values)
y_proba = best_model.predict_proba(X_val.values)[:, 1]


In [213]:
# save the model
best_model.save_model(DATA_PATH + "simple_xgb_model.json")
# save the feature names so R can pick them up
with open(DATA_PATH + "simple_features.json", "w") as f:
    json.dump(list(X_tr_drop.columns), f)

In [214]:
# load model 
best_model = XGBClassifier()
best_model.load_model(DATA_PATH + "simple_xgb_model.json")

## Evaluation

### Evaluation set

In [ ]:
# evaluation df for training data
def move_column_inplace(df, col, pos):
    col = df.pop(col)
    df.insert(pos, col.name, col)

y_pred_val = best_model.predict_proba(X_val)[:, 1]
y_pred_thresh = (y_pred_val >= THRESHOLD).astype(int)
y_prob_val = best_model.predict_proba(X_val)[:, 1]

val_eval_df = df_tr.copy()

val_eval_df = val_eval_df[val_eval_df['henkilotunnus'].isin(group_val.tolist())]
val_eval_df['original_y'] = y_val
val_eval_df['pred_proba'] = y_prob_val
val_eval_df['y_pred'] = y_pred_thresh
move_column_inplace(val_eval_df, 'pred_proba', 1)
move_column_inplace(val_eval_df, 'original_y', 1)
move_column_inplace(val_eval_df, 'y_pred', 1)
move_column_inplace(val_eval_df, 'naytteenotto_hetki', 5)
cols = X_tr_drop.columns
cols = np.concatenate([['henkilotunnus', 'infektion_binary', 'original_y', 'y_pred', 'pred_proba', 'naytteenotto_hetki', 'sykli_IND', 'cycle_number', 'age', 'p_crp'], cols])
val_eval_df = val_eval_df[cols]
val_eval_df.rename(columns={'original_y':'y_original'}).to_csv(DATA_PATH + 'val_eval_df.csv')

### Training set

In [ ]:
print('Training set:')
y_pred_train = best_model.predict_proba(X_train)[:, 1]
y_pred_thresh = (y_pred_train >= THRESHOLD).astype(int)

kappa = cohen_kappa_score(y_train, y_pred_thresh)
recall = recall_score(y_train, y_pred_thresh)
precision = precision_score(y_train, y_pred_thresh)
f1 = f1_score(y_train, y_pred_thresh)
accuracy = accuracy_score(y_train, y_pred_thresh)

print('Kappa:', kappa)
print("Recall:", recall)
print("Precision:", precision)
print("F1 score:", f1)
customf1 = custom_f1(y_train, y_pred_thresh)
print("Custom F1:", customf1)


In [ ]:
# confusion matrix
y_prob_train = best_model.predict_proba(X_train)[:, 1]

# confusion matrix with changed threshold
y_pred_thresh = (y_prob_train >= THRESHOLD).astype(int)
plot_confusion_matrix_with_percentages(
    y_train, y_pred_thresh, 
    percent_type='row', 
    threshold=THRESHOLD,
    save_path = PLOT_PATH+'simple_model/train_cm',  # <-- save figure here
    title='Training set',
    cmap=colourmap,
    text_color='white',
    plot_data=True
)

kappa = cohen_kappa_score(y_train, y_pred_thresh)
recall = recall_score(y_train, y_pred_thresh)
precision = precision_score(y_train, y_pred_thresh)
f1 = f1_score(y_train, y_pred_thresh)
accuracy = accuracy_score(y_train, y_pred_thresh)
customf1 = custom_f1(y_train, y_pred_thresh)


row = pd.DataFrame({'data':[f'simple_train'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 
                    'specificity':[recall_score(y_train, y_pred_thresh, pos_label=0)], 'PNV':[calc_pnv(y_train, y_pred_thresh)]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)

print(f"Kappa      : {kappa:.4f}") # Is my model better than random, and by how much?
print(f"Recall     : {recall:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"F1 score   : {f1:.4f}")
print(f"Custom F1  : {customf1:.4f}")


In [ ]:
def _row_metrics(y, yh):
    return {
        "F1_score":    f1_score(y, yh, zero_division=0),
        "kappa":       cohen_kappa_score(y, yh),
        "recall":      recall_score(y, yh, zero_division=0),
        "precision":   precision_score(y, yh, zero_division=0),
        "accuracy":    accuracy_score(y, yh),
        "customf1":    custom_f1(y, yh),
        "specificity": recall_score(y, yh, pos_label=0, zero_division=0),
        "PNV":         calc_pnv(y, yh),
    }

def add_metrics_column(table, y_true, y_prob, thr, name, groups=None, extra=None,
                       n_boot=2000, seed=42, alpha=0.05, add_auc=True):
    """Append one run to the metrics table as the column group (name, est/lo/hi).
       table=None starts a new table. Re-using a name replaces that run."""
    y = np.asarray(y_true).astype(int); p = np.asarray(y_prob, float)

    def _all(yy, pp):
        m = _row_metrics(yy, (pp >= thr).astype(int))
        if add_auc:
            m = {"AUROC": roc_auc_score(yy, pp),
                 "AUPRC": average_precision_score(yy, pp), **m}
        if extra:
            for k, fn_ in extra.items():
                m[k] = fn_(yy, (pp >= thr).astype(int))
        return m

    point = _all(y, p)
    keys = list(point)

    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    _, inv = np.unique(g, return_inverse=True)
    idx = [np.flatnonzero(inv == k) for k in range(inv.max() + 1)]

    rng = np.random.default_rng(seed)
    boots = np.full((n_boot, len(keys)), np.nan)
    for b in range(n_boot):
        s = rng.integers(0, len(idx), len(idx))
        ii = np.concatenate([idx[j] for j in s])
        yb, pb = y[ii], p[ii]
        if yb.min() == yb.max():
            continue
        m = _all(yb, pb)
        boots[b] = [m[k] for k in keys]

    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    col = pd.DataFrame(
        {(name, "est"): [point[k] for k in keys],
         (name, "lo"):  np.nanpercentile(boots, lo_q, axis=0),
         (name, "hi"):  np.nanpercentile(boots, hi_q, axis=0)},
        index=pd.Index(keys, name="metric"))
    col.columns = pd.MultiIndex.from_tuples(col.columns, names=["run", "stat"])

    if table is None:
        return col
    if name in table.columns.get_level_values("run"):
        table = table.drop(columns=name, level="run")
    order = list(table.index) + [k for k in col.index if k not in table.index]
    return table.join(col, how="outer").reindex(order)


def fmt_table(df, decimals=2):
    """Collapse each run's est/lo/hi into one 'est (lo–hi)' column."""
    out = pd.DataFrame(index=df.index)
    for r in df.columns.get_level_values("run").unique():
        e, lo, hi = df[(r, "est")], df[(r, "lo")], df[(r, "hi")]
        out[r] = [f"{a:.{decimals}f} ({b:.{decimals}f}–{c:.{decimals}f})"
                  if pd.notna(a) else "" for a, b, c in zip(e, lo, hi)]
    return out

metrics_df_ci = add_metrics_column(None, y_train, y_prob_train,  THRESHOLD, "train_top")
fmt_table(metrics_df_ci)

#### AUROC and AUPRC with 95% CI

In [ ]:
#y_train, y_prob_train
DCA_ORANGE = "#d95f02"
DCA_BLUE = "#2b6cb0"
DCA_BAND = "#fde68a"
DCA_GRAY = "#6b7280"
DCA_INK = "#111827"


def plot_curve_ci(y_true, y_prob, kind="roc", groups=None, n_boot=1000,
                  seed=42, alpha=0.05, n_grid=201, ax=None, title=None,
                  figsize=(7.0, 7.0),
                  plot_data=False, plot_data_name=None):     # <-- NEW
    """ROC or precision-recall curve with a bootstrap 95% band and AUC (95% CI)."""
    y = np.asarray(y_true).astype(int); p = np.asarray(y_prob, float)
    grid = np.linspace(0, 1, n_grid)

    def _curve(yy, pp):
        if kind == "roc":
            fpr, tpr, _ = roc_curve(yy, pp)
            return np.interp(grid, fpr, tpr), roc_auc_score(yy, pp)
        pr, rc, _ = precision_recall_curve(yy, pp)
        return np.interp(grid, rc[::-1], pr[::-1]), average_precision_score(yy, pp)

    y_pt, auc_pt = _curve(y, p)

    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    _, inv = np.unique(g, return_inverse=True)
    idx = [np.flatnonzero(inv == k) for k in range(inv.max() + 1)]

    rng = np.random.default_rng(seed)
    curves = np.full((n_boot, n_grid), np.nan)
    aucs = np.full(n_boot, np.nan)
    for b in range(n_boot):
        s = rng.integers(0, len(idx), len(idx))
        ii = np.concatenate([idx[j] for j in s])
        yb, pb = y[ii], p[ii]
        if yb.min() == yb.max():
            continue
        curves[b], aucs[b] = _curve(yb, pb)

    lo_q, hi_q = 100 * alpha / 2, 100 * (1 - alpha / 2)
    band_lo = np.nanpercentile(curves, lo_q, axis=0)
    band_hi = np.nanpercentile(curves, hi_q, axis=0)
    a_lo, a_hi = np.nanpercentile(aucs, lo_q), np.nanpercentile(aucs, hi_q)

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    name = "AUROC" if kind == "roc" else "AUPRC"
    ax.fill_between(grid, band_lo, band_hi, color=DCA_BLUE, alpha=0.20, lw=0, zorder=1)
    ax.plot(grid, y_pt, color=DCA_BLUE, lw=2, zorder=3)

    if kind == "roc":
        ax.plot([0, 1], [0, 1], color=DCA_GRAY, ls="--", lw=1, zorder=2)
        ax.set_xlabel("False positive rate", fontsize=14)
        ax.set_ylabel("True positive rate", fontsize=14)
        curve_lbl, loc = "ROC curve", "lower right"
    else:
        base = y.mean()
        ax.axhline(base, color=DCA_GRAY, ls="--", lw=1, zorder=2)
        ax.set_xlabel("Recall", fontsize=14)
        ax.set_ylabel("Precision", fontsize=14)
        curve_lbl, loc = "Precision–recall curve", "upper right"

    _line = Line2D([], [], color=DCA_BLUE, lw=2)
    _band = Patch(facecolor=DCA_BLUE, alpha=0.20, lw=0)
    _txt  = Line2D([], [], ls="none", marker="none")
    _base = Line2D([], [], color=DCA_GRAY, ls="--", lw=1)

    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.1)
    #ax.legend(fontsize=11, loc=loc, frameon=False)

    line = Line2D([], [], color=DCA_BLUE, lw=2)
    _band = Patch(facecolor=DCA_BLUE, alpha=0.20, lw=0)
    _txt  = Line2D([], [], ls="none", marker="none")     # text-only row
    _auc  = f"{name} = {auc_pt:.2f} (95% CI {a_lo:.2f}–{a_hi:.2f})"

    if kind == "roc":
        _h = [_line, _band, _txt]
        _l = [curve_lbl, "Curve – 95% CI", _auc]
    else:
        _base = Line2D([], [], color=DCA_GRAY, ls="--", lw=1)
        _h = [_line, _band, _txt]
        _l = [curve_lbl, "Curve – 95% CI", _auc]

    ax.legend(_h, _l, fontsize=11, loc=loc, frameon=False)
    ax.set_title(title or ("ROC curve" if kind == "roc" else "Precision-recall curve"),
                 fontsize=16)
    ax.set_box_aspect(1)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    # ---- source data for the journal --------------------------------------
    if plot_data:
        import pandas as pd
        if kind == "roc":
            _curve_df = pd.DataFrame({"false_positive_rate": grid,
                                      "true_positive_rate": y_pt,
                                      "tpr_ci_lower": band_lo,
                                      "tpr_ci_upper": band_hi})
            _ref = np.nan
        else:
            _curve_df = pd.DataFrame({"recall": grid,
                                      "precision": y_pt,
                                      "precision_ci_lower": band_lo,
                                      "precision_ci_upper": band_hi})
            _ref = float(base)
        _summary = pd.DataFrame([{
            "metric": name,
            "value": float(auc_pt),
            "ci_lower": float(a_lo),
            "ci_upper": float(a_hi),
            "ci_level": 1 - alpha,
            "n_bootstrap": int(n_boot),
            "n_bootstrap_used": int(np.isfinite(aucs).sum()),
            "bootstrap_unit": "observation" if groups is None else "group",
            "seed": int(seed),
            "n_total": int(y.size),
            "n_positive": int(y.sum()),
            "prevalence": float(y.mean()),
            "reference_line": _ref,
        }])
        write_plot_data(
            plot_data_name or ("roc_curve" if kind == "roc" else "pr_curve"),
            {"curve": _curve_df, "summary": _summary},
            out_dir=PLOT_DATA_PATH,
            note=(f"{name}={auc_pt:.4f} (95% CI {a_lo:.4f}-{a_hi:.4f}); "
                  f"n_boot={n_boot}; grid={n_grid} points"),
        )

    print(f"{name} {auc_pt:.3f} (95% CI {a_lo:.3f}-{a_hi:.3f})")
    return fig, (auc_pt, a_lo, a_hi)


ROC_NAME = 'train_ROC_curve_CI95'
fig, roc_ci  = plot_curve_ci(y_train, y_prob_train, kind="roc", groups=None,
                             plot_data=True, plot_data_name=ROC_NAME)
fig.savefig(PLOT_PATH + 'simple_model/' + ROC_NAME, dpi=300, bbox_inches='tight')
plt.show()

PR_NAME = 'train_RP_curve_CI95'
fig, prc_ci = plot_curve_ci(y_train, y_prob_train, kind="pr", groups=None,
                            plot_data=True, plot_data_name=PR_NAME)
fig.savefig(PLOT_PATH + 'simple_model/' + PR_NAME, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(PLOT_PATH + 'train_RP_curve_CI95')

#### One-day-early correction

In [ ]:
y_prob_train = best_model.predict_proba(X_train)[:, 1]

df, score, y_pred_mod = custom_score(y_train, y_prob_train, group_train, threshold=THRESHOLD, eval=True)

# Evaluation metrics
f1 = f1_score(y_train, y_pred_mod)
recall = recall_score(y_train, y_pred_mod)
accuracy = accuracy_score(y_train, y_pred_mod)
precision = precision_score(y_train, y_pred_mod)
kappa = cohen_kappa_score(y_train, y_pred_mod)
customf1 = custom_f1(y_train, y_pred_mod)

print(f"Kappa       : {kappa:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
customf1 = custom_f1(y_train, y_pred_mod)
print(f"Custom F1   : {customf1:.4f}")

# Confusion matrix
plot_confusion_matrix_with_percentages(
    y_train, y_pred_mod, 
    percent_type='row', 
    threshold=THRESHOLD,
    save_path= PLOT_PATH+'/simple_model/train_one_day_early_correction_cm',
    # title='Training set\nConfusion Matrix with Row %',
    title='Training set',
    cmap='coolwarm',
    text_color='white'

)

row = pd.DataFrame({'data':[f'simple_train_corrected'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 
                    'specificity':[recall_score(y_train, y_pred_mod, pos_label=0)], 'PNV':[calc_pnv(y_train, y_pred_mod)]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)
metrics_df

print(f"Kappa      : {kappa:.4f}") # Is my model better than random, and by how much?
print(f"Recall     : {recall:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"F1 score   : {f1:.4f}")
print(f"Custom F1  : {customf1:.4f}")

metrics_df_ci = add_metrics_column(metrics_df_ci, y_train, y_pred_mod,  THRESHOLD, "train_top_corrected")
fmt_table(metrics_df_ci)


In [ ]:
# evaluation df for training data
def move_column_inplace(df, col, pos):
    col = df.pop(col)
    df.insert(pos, col.name, col)

train_eval_df = df_tr.copy()
train_eval_df = train_eval_df[train_eval_df['henkilotunnus'].isin(group_train)]
train_eval_df['original_y'] = y_train
train_eval_df['pred_proba'] = y_prob_train
train_eval_df['y_pred'] = y_pred_thresh
move_column_inplace(train_eval_df, 'pred_proba', 1)
move_column_inplace(train_eval_df, 'original_y', 1)
move_column_inplace(train_eval_df, 'y_pred', 1)
move_column_inplace(train_eval_df, 'naytteenotto_hetki', 5)

cols = X_tr_drop.columns
cols = np.concatenate([['henkilotunnus', 'infektion_binary', 'original_y', 'y_pred', 'pred_proba', 'naytteenotto_hetki', 'sykli_IND', 'cycle_number', 'age', 'p_crp'], cols])
train_eval_df = train_eval_df[cols]
train_eval_df.rename(columns={'original_y':'y_original'}).to_csv(DATA_PATH + 'train_eval_df.csv')

In [ ]:
DATA_PATH + 'train_eval_df.csv'

### Test set

In [ ]:

print('\nTest set:')
y_pred_test = best_model.predict_proba(X_test_drop)[:, 1]
y_pred_thresh = (y_pred_test >= THRESHOLD).astype(int)

kappa = cohen_kappa_score(y_test, y_pred_thresh)
recall = recall_score(y_test, y_pred_thresh)
precision = precision_score(y_test, y_pred_thresh)
f1 = f1_score(y_test, y_pred_thresh)
accuracy = accuracy_score(y_test, y_pred_thresh)
customf1 = custom_f1(y_test, y_pred_thresh)

row = pd.DataFrame({'data':[f'simple_test_0.5'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1]})

print('Kappa:', kappa)
print("Recall:", recall)
print("Precision:", precision)
print("F1 score:", f1)
print(f"Custom F1   : {customf1:.4f}")



In [ ]:
#THRESHOLD = 0.2
THRESHOLD

In [ ]:

# ROC curve
y_prob_test = best_model.predict_proba(X_test_drop)[:, 1]

# confusion matrix with changed threshold
y_pred_thresh = (y_prob_test >= THRESHOLD).astype(int)
plot_confusion_matrix_with_percentages(
    y_test, y_pred_thresh, 
    percent_type='row', 
    threshold=THRESHOLD,
    save_path = PLOT_PATH+'simple_model/test_cm',  # <-- save figure here
    title='Test set',
    cmap=colourmap,
    text_color='white',
    plot_data=True
)

kappa = cohen_kappa_score(y_test, y_pred_thresh)
recall = recall_score(y_test, y_pred_thresh)
precision = precision_score(y_test, y_pred_thresh)
f1 = f1_score(y_test, y_pred_thresh)
accuracy = accuracy_score(y_test, y_pred_thresh)
customf1 = custom_f1(y_test, y_pred_thresh)

row = pd.DataFrame({'data':[f'simple_test'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 
                    'specificity':[recall_score(y_test, y_pred_thresh, pos_label=0)], 'PNV':[calc_pnv(y_test, y_pred_thresh)]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)


print(f"Kappa      : {kappa:.4f}") # Is my model better than random, and by how much?
print(f"Recall     : {recall:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"F1 score   : {f1:.4f}")
print(f"Custom F1  : {customf1:.4f}")

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_thresh,  THRESHOLD, "test_top")
fmt_table(metrics_df_ci)



#### AUROC and AUPRC with 95% CI

In [ ]:
fig, roc_ci  = plot_curve_ci(y_test, y_prob_test, kind="roc", groups=None, plot_data=True, plot_data_name="test_ROC_curve_CI95")
fig.savefig(PLOT_PATH + 'simple_model/test_ROC_curve_CI95', dpi=300, bbox_inches='tight')
plt.show()

fig, prc_ci = plot_curve_ci(y_test, y_prob_test, kind="pr", groups=None, plot_data=True, plot_data_name="test_PR_curve_CI95")
fig.savefig(PLOT_PATH + 'simple_model/test_RP_curve_CI95', dpi=300, bbox_inches='tight')
plt.show()

#### One-day-early correction

In [ ]:
y_prob_test = best_model.predict_proba(X_test_drop)[:, 1]

df, score, y_pred_mod = custom_score(y_test, y_prob_test, group_test, threshold=THRESHOLD, eval=True)

# 👉 3. Evaluation metrics
f1 = f1_score(y_test, y_pred_mod)
recall = recall_score(y_test, y_pred_mod)
accuracy = accuracy_score(y_test, y_pred_mod)
precision = precision_score(y_test, y_pred_mod)
kappa = cohen_kappa_score(y_test, y_pred_mod)

print(f"Kappa       : {kappa:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
customf1 = custom_f1(y_test, y_pred_mod)
print(f"Custom F1   : {customf1:.4f}")


# 👉 4. Confusion matrix
plot_confusion_matrix_with_percentages(
    y_test, y_pred_mod, 
    percent_type='row', 
    threshold=THRESHOLD,
    save_path = PLOT_PATH+'simple_model/test_set_cm_correction', # <-- save figure here
    title='Test set',
    cmap='coolwarm',
    text_color='white'
)

row = pd.DataFrame({'data':[f'simple_test_corrected'], 'F1_score':[f1], 'kappa':[kappa], 'recall':[recall], 'precision':[precision], 'accuracy':[accuracy], 'customf1':[customf1], 
                    'specificity':[recall_score(y_test, y_pred_mod, pos_label=0)], 'PNV':[calc_pnv(y_test, y_pred_mod)]})
metrics_df = pd.concat([metrics_df, row], ignore_index=True)
metrics_df

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "test_top_corrected")
fmt_table(metrics_df_ci)

## Test data visualisations

In [230]:
df_test = pd.read_csv(DATA_PATH + 'df_test.csv').drop(columns='Unnamed: 0')

In [231]:
def move_column_inplace(df, col, pos):
    col = df.pop(col)
    df.insert(pos, col.name, col)

### Evaluation dataframe

In [232]:
# all together
eval_df = df_test.copy()
eval_df['pred_modified_y'] = y_pred_mod
eval_df['original_y'] = y_test
eval_df['pred_proba'] = y_prob_test
eval_df['y_pred'] = y_pred_thresh
move_column_inplace(eval_df, 'pred_proba', 1)
move_column_inplace(eval_df, 'pred_modified_y', 1)
move_column_inplace(eval_df, 'original_y', 1)
move_column_inplace(eval_df, 'y_pred', 1)
move_column_inplace(eval_df, 'naytteenotto_hetki', 5)

cols = X_tr_drop.columns
cols = np.concatenate([['henkilotunnus', 'infektion_binary', 'original_y', 'y_pred', 'pred_modified_y', 'pred_proba', 'naytteenotto_hetki', 'sykli_IND', 'cycle_number', 'age', 'p_crp'], cols])
eval_df = eval_df[cols]
eval_df.rename(columns={'original_y':'y_original'}).to_csv(DATA_PATH + 'eval_df.csv')

In [233]:
# misclassified rows
eval_df[eval_df['infektion_binary']==eval_df['y_pred']].to_csv(DATA_PATH + 'TP_TN_df.csv')

### Decision curve analysis

In [ ]:

# --- standardise to the column names used in all later steps ---
dca_df = (eval_df
          .rename(columns={"henkilotunnus": "patient_id",
                           "pred_proba":    "p",
                           "original_y":    "y"})
          [["patient_id", "y", "p"]]
          .copy())
dca_df["y"] = dca_df["y"].astype(int)
dca_df["p"] = dca_df["p"].astype(float)

# --- Step 2: the counts that go in the manuscript ---
n_patients = dca_df["patient_id"].nunique()
n_days     = len(dca_df)
n_events   = int(dca_df["y"].sum())
prevalence = dca_df["y"].mean()

days_per_pat = dca_df.groupby("patient_id").size()
ev_per_pat   = dca_df.groupby("patient_id")["y"].sum()

print(f"Patients                      : {n_patients}")
print(f"Patient-days                  : {n_days}")
print(f"FN-positive patient-days      : {n_events}")
print(f"Prevalence per patient-day    : {prevalence:.4f}  ({prevalence*100:.2f} %)")
print(f"Patients with >=1 FN day      : {(ev_per_pat > 0).sum()} "
      f"({(ev_per_pat > 0).mean()*100:.1f} % of patients)")
print(f"Days per patient, median (IQR): {days_per_pat.median():.0f} "
      f"({days_per_pat.quantile(.25):.0f}-{days_per_pat.quantile(.75):.0f}), "
      f"range {days_per_pat.min()}-{days_per_pat.max()}")

# --- sanity checks: catch data problems before they become figure problems ---
print("\n--- sanity checks ---")
print("missing values       :", dca_df.isna().sum().to_dict())
print("y values present     :", sorted(dca_df["y"].unique()))
print("p range              :", f"{dca_df['p'].min():.4f} - {dca_df['p'].max():.4f}")
print("p outside [0,1]      :", int(((dca_df["p"] < 0) | (dca_df["p"] > 1)).sum()))
print("exact duplicate rows :", int(dca_df.duplicated().sum()))
print("mean predicted risk  :", f"{dca_df['p'].mean():.4f}", "| observed:", f"{prevalence:.4f}")

In [ ]:
# calibration (again)

def dca_logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return np.log(p / (1 - p))

_y  = dca_df["y"].to_numpy(int)
_p  = dca_df["p"].to_numpy(float)
_lp = dca_logit(_p)
_grp = dca_df["patient_id"].to_numpy()

# --- calibration intercept & slope, cluster-robust SEs by patient ---
try:
    import statsmodels.api as sm
    _fit = sm.GLM(_y, sm.add_constant(_lp), family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": _grp})
    dca_cal_intercept, dca_cal_slope = _fit.params
    _ci = _fit.conf_int()
    _int_lo, _int_hi     = _ci[0]
    _slope_lo, _slope_hi = _ci[1]

    # calibration-in-the-large: intercept with the slope pinned at 1
    _fit0 = sm.GLM(_y, np.ones((len(_y), 1)), family=sm.families.Binomial(),
                   offset=_lp).fit(cov_type="cluster", cov_kwds={"groups": _grp})
    _citl = _fit0.params[0]
    _citl_lo, _citl_hi = _fit0.conf_int()[0]
except ImportError:
    from sklearn.linear_model import LogisticRegression
    _lr = LogisticRegression(C=1e12, solver="lbfgs", max_iter=1000).fit(_lp.reshape(-1, 1), _y)
    dca_cal_intercept, dca_cal_slope = float(_lr.intercept_[0]), float(_lr.coef_[0][0])
    _int_lo = _int_hi = _slope_lo = _slope_hi = np.nan
    _citl = _citl_lo = _citl_hi = np.nan

_prev  = _y.mean()
_brier = float(np.mean((_p - _y) ** 2))
_brier_scaled = 1 - _brier / (_prev * (1 - _prev))

print(f"Calibration slope        : {dca_cal_slope:.3f}  [{_slope_lo:.3f}, {_slope_hi:.3f}]   (ideal 1)")
print(f"Calibration intercept    : {dca_cal_intercept:+.3f} [{_int_lo:+.3f}, {_int_hi:+.3f}]  (ideal 0)")
print(f"Calibration-in-the-large : {_citl:+.3f} [{_citl_lo:+.3f}, {_citl_hi:+.3f}]  (ideal 0)")
print(f"Brier score              : {_brier:.5f}")
print(f"Scaled Brier (skill)     : {_brier_scaled:.3f}   (0 = no better than predicting prevalence)")
print(f"Mean predicted {_p.mean():.4f} vs observed {_prev:.4f}  (ratio {_p.mean()/_prev:.2f})")

In [ ]:
def plot_calibration_full(df, n_bins=10, show_lowess=True, xmax=1.0, ymax=None,
                          ax=None, equal_aspect=False,
                          plot_data=False, plot_data_name=None):     # <-- NEW    
    y = df["y"].to_numpy(int); p = df["p"].to_numpy(float)
    lp = dca_logit(p); grp = df["patient_id"].to_numpy()

    # the calibration model itself supplies the smooth curve and its band,
    # with cluster-robust covariance so the band respects patient clustering
    fit = sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": grp})
    gx = np.linspace(1e-4, xmax, 400)
    pr = fit.get_prediction(sm.add_constant(dca_logit(gx))).summary_frame(alpha=0.05)

    _b = pd.qcut(p, n_bins, labels=False, duplicates="drop")
    g = (pd.DataFrame({"p": p, "y": y, "bin": _b}).groupby("bin")
           .agg(mean_p=("p","mean"), obs=("y","mean"), n=("y","size"), ev=("y","sum")))
    _lo = np.nan_to_num(beta.ppf(0.025, g["ev"],     g["n"]-g["ev"]+1), nan=0.0)
    _hi = np.nan_to_num(beta.ppf(0.975, g["ev"] + 1, g["n"]-g["ev"]),   nan=1.0)
    _yerr = np.clip(np.vstack([g["obs"] - _lo, _hi - g["obs"]]), 0, None)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7.0, 7.6))
    else:
        fig = ax.figure

    ax.plot([0, xmax], [0, xmax], color="grey", ls="--", lw=1, zorder=2,
            label="Ideal calibration")
    ax.fill_between(gx, pr["mean_ci_lower"], pr["mean_ci_upper"],
                    color="tab:blue", alpha=0.2, lw=0, zorder=1, label="Curve - 95% confidence interval")
    ax.plot(gx, pr["mean"], color="tab:blue", lw=2, zorder=4, label="Calibration curve")
    if show_lowess:
        try:
            from statsmodels.nonparametric.smoothers_lowess import lowess
            _sup = np.quantile(p, 0.995)              # don't draw beyond the data
            _s = lowess(y, p, frac=0.6, it=0, return_sorted=True)
            _m = _s[:, 0] <= min(xmax, _sup)
            #ax.plot(_s[_m, 0], _s[_m, 1], color="tab:blue", lw=1.2, ls=(0, (3, 2)),alpha=.75, zorder=3, label="Flexible (lowess)")
        except ImportError:
            pass
     # 2. label= removed from this call, and the two proxy handles added under it
    ax.errorbar(g["mean_p"], g["obs"], yerr=_yerr, fmt="o", ms=5, lw=1, capsize=3,
                color="tab:blue", zorder=5)
    _pt = Line2D([], [], color="tab:blue", marker="o", ls="none", ms=5)
    _ci = ax.errorbar([np.nan], [np.nan], yerr=[[1.0], [1.0]], fmt="none",
                      ecolor="tab:blue", elinewidth=1, capsize=3)


    ax.set_xlim(0, xmax); ax.set_ylim(0, ymax)
    ax.set_xlabel("Predicted probability", fontsize=14)
    ax.set_ylabel("Observed FN rate", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.1)
    #ax.legend(fontsize=11, loc="upper left", frameon=False)
    _h, _l = ax.get_legend_handles_labels()
    _by = dict(zip(_l, _h))
    ax.legend([_pt, _ci, _by["Calibration curve"],
               _by["Curve - 95% confidence interval"], _by["Ideal calibration"]],
              ["Groups", "Group - SD", "Calibration curve",
               "Curve - 95% CI", "Ideal calibration"],
              fontsize=11, loc="upper left", frameon=False)
    ax.set_title(f"Calibration plot",
                 fontsize=16)
    print(f"slope {fit.params[1]:.2f}, intercept {fit.params[0]:+.2f}")
    if equal_aspect:
        ax.set_aspect("equal", adjustable="box")
    else:
        ax.set_box_aspect(1)  
    ax.spines[["top", "right"]].set_visible(False)

    _ax2 = ax.inset_axes([0, -0.36, 1, 0.20])
    #_ax2.hist(p, bins=80, range=(0, xmax), color="#94a3b8")
    _hist_n, _hist_edges, _ = _ax2.hist(p, bins=80, range=(0, xmax), color="#94a3b8")
    _ax2.set_yticks([]); _ax2.set_xlim(0, xmax)
    _ax2.set_xlabel("Distribution of predictions", fontsize=10)
    _ax2.tick_params(labelsize=10)
    _ax2.spines[["top", "right", "left"]].set_visible(False)

    fig.tight_layout()

    # ---- source data for the journal --------------------------------------
    if plot_data:
        curve = pd.DataFrame({
            "predicted_probability": gx,
            "observed_rate":         pr["mean"].to_numpy(),
            "observed_ci_lower":     pr["mean_ci_lower"].to_numpy(),
            "observed_ci_upper":     pr["mean_ci_upper"].to_numpy(),
        })
        bins = pd.DataFrame({
            "bin":                        g.index.to_numpy(),
            "mean_predicted_probability": g["mean_p"].to_numpy(),
            "observed_rate":              g["obs"].to_numpy(),
            "observed_ci_lower":          g["obs"].to_numpy() - _yerr[0],
            "observed_ci_upper":          g["obs"].to_numpy() + _yerr[1],
            "n":                          g["n"].to_numpy(),
            "n_events":                   g["ev"].to_numpy(),
        })
        hist = pd.DataFrame({
            "bin_left":  _hist_edges[:-1],
            "bin_right": _hist_edges[1:],
            "count":     _hist_n.astype(int),
        })
        _b0, _b1 = np.asarray(fit.params, float)[:2]
        _ci95 = np.asarray(fit.conf_int(alpha=0.05), float)
        _se = np.asarray(fit.bse, float)
        summary = pd.DataFrame([{
            "calibration_intercept": _b0,
            "intercept_se":          _se[0],
            "intercept_ci_lower":    _ci95[0, 0],
            "intercept_ci_upper":    _ci95[0, 1],
            "calibration_slope":     _b1,
            "slope_se":              _se[1],
            "slope_ci_lower":        _ci95[1, 0],
            "slope_ci_upper":        _ci95[1, 1],
            "n_total":               int(y.size),
            "n_events":              int(y.sum()),
            "n_clusters":            int(pd.unique(grp).size),
            "cov_type":              "cluster (patient_id)",
            "n_bins":                int(g.shape[0]),
        }])
        write_plot_data(
            plot_data_name or "calibration_plot",
            {"curve": curve, "bins": bins, "summary": summary,
             "prediction_histogram": hist},
            out_dir=PLOT_DATA_PATH,
            note=(f"slope={_b1:.4f}, intercept={_b0:+.4f}; "
                  f"cluster-robust SE on patient_id; "
                  f"curve on {len(gx)}-point grid; {g.shape[0]} bins; "
                  f"inset histogram {len(_hist_n)} bins"),
        )

    return fig, fit, g.assign(ci_lo=_lo, ci_hi=_hi)

CAL_NAME = 'calibration_plot_DCA'
fig, cal_fit, cal_table = plot_calibration_full(dca_df, plot_data=True,
                                                plot_data_name=CAL_NAME)
fig.savefig(PLOT_PATH+'simple_model/'+CAL_NAME, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Step 4: choose and justify the plausible threshold range ---
dca_range_lo, dca_range_hi = 0.02, 0.20     # <-- the decision; revise after clinical input

_y = dca_df["y"].to_numpy(int)
_p = dca_df["p"].to_numpy(float)
_prev = _y.mean()

_show = [0.02, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]
_rows = []
for _pt in _show:
    _alert = _p >= _pt
    _tp = int((_alert & (_y == 1)).sum())
    _fp = int((_alert & (_y == 0)).sum())
    _fn = int((~_alert & (_y == 1)).sum())
    _ppv = _tp / (_tp + _fp) if (_tp + _fp) else np.nan
    _rows.append({
        "threshold":               _pt,
        "FP tolerated per TP":     _pt / (1 - _pt),
        "alerts per 100 pt-days":  _alert.mean() * 100,
        "sensitivity":             _tp / (_tp + _fn) if (_tp + _fn) else np.nan,
        "PPV":                     _ppv,
        "number needed to alert":  (_tp + _fp) / _tp if _tp else np.nan,
        "net benefit > 0?":        "yes" if _ppv > _pt else "no",
    })

dca_threshold_table = pd.DataFrame(_rows).set_index("threshold")
print(f"prevalence per patient-day = {_prev:.4f}\n")
print(dca_threshold_table.round(3).to_string())

In [ ]:
def dca_net_benefit(y, p, thresholds):
    """Net benefit of the model and of 'treat all' at each threshold.
    NB = TP/n - (FP/n) * pt/(1-pt).  'Treat none' is 0 by definition.
    Note TP and FP are divided by ALL patient-days, not by the alerts."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    n, prev = len(y), y.mean()
    nb_model = np.empty(len(thresholds))
    nb_all   = np.empty(len(thresholds))
    for i, pt in enumerate(thresholds):
        w = pt / (1.0 - pt)
        alert = p >= pt
        tp = np.sum(alert & (y == 1)) / n
        fp = np.sum(alert & (y == 0)) / n
        nb_model[i] = tp - fp * w
        nb_all[i]   = prev - (1.0 - prev) * w
    return nb_model, nb_all

dca_thresholds = np.round(np.arange(0.01, 1.0, 0.005), 4)
_nb_model, _nb_all = dca_net_benefit(dca_df["y"], dca_df["p"], dca_thresholds)

dca_curve = pd.DataFrame({
    "threshold":     dca_thresholds,
    "nb_model":      _nb_model,
    "nb_treat_all":  _nb_all,
    "nb_treat_none": 0.0,
})
dca_curve["best_default"] = dca_curve[["nb_treat_all", "nb_treat_none"]].max(axis=1)
dca_curve["nb_gain_vs_best_default"] = dca_curve["nb_model"] - dca_curve["best_default"]

_show = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
print(dca_curve[dca_curve["threshold"].isin(_show)].round(4).to_string(index=False))

_pos = dca_curve.loc[dca_curve["nb_model"] > 0, "threshold"]
_win = dca_curve.loc[dca_curve["nb_gain_vs_best_default"] > 0, "threshold"]
print(f"\nModel NB > 0 for thresholds        {_pos.min():.3f}-{_pos.max():.3f}")
print(f"Model beats BOTH defaults for      {_win.min():.3f}-{_win.max():.3f}")
_in = dca_curve["threshold"].between(dca_range_lo, dca_range_hi)
print(f"Within {dca_range_lo}-{dca_range_hi}: gain over best default "
      f"{dca_curve.loc[_in, 'nb_gain_vs_best_default'].min():+.4f} to "
      f"{dca_curve.loc[_in, 'nb_gain_vs_best_default'].max():+.4f}")


In [ ]:
_w = dca_thresholds / (1 - dca_thresholds)
dca_curve["net_reduction_per_100"] = (dca_curve["nb_model"] - dca_curve["nb_treat_all"]) / _w * 100
dca_curve["alerts_per_100"] = [(dca_df["p"].to_numpy() >= t).mean() * 100 for t in dca_thresholds]

_show = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
print(dca_curve[dca_curve["threshold"].isin(_show)][
    ["threshold", "nb_model", "nb_treat_all", "nb_gain_vs_best_default",
     "alerts_per_100", "net_reduction_per_100"]].round(3).to_string(index=False))

In [ ]:
def _dca_nb_fast(y, p, th):
    n, prev = len(y), y.mean()
    p1, p0 = np.sort(p[y == 1]), np.sort(p[y == 0])
    tp = (len(p1) - np.searchsorted(p1, th, side="left")) / n
    fp = (len(p0) - np.searchsorted(p0, th, side="left")) / n
    w = th / (1 - th)
    return tp - fp * w, prev - (1 - prev) * w

def dca_bootstrap(df, th, n_boot=2000, seed=42):
    """Resample PATIENTS with replacement, keeping each patient's full day-sequence."""
    rng = np.random.default_rng(seed)
    _g = {pid: (s["y"].to_numpy(int), s["p"].to_numpy(float))
          for pid, s in df.groupby("patient_id")}
    _pids = np.array(list(_g.keys()))
    m_nb   = np.empty((n_boot, len(th)))
    m_gain = np.empty((n_boot, len(th)))
    for b in range(n_boot):
        s = rng.choice(_pids, size=len(_pids), replace=True)
        _yb = np.concatenate([_g[i][0] for i in s])
        _pb = np.concatenate([_g[i][1] for i in s])
        nb, na = _dca_nb_fast(_yb, _pb, th)
        m_nb[b]   = nb
        m_gain[b] = nb - np.maximum(na, 0.0)      # gain over the better default
    return m_nb, m_gain

_m_nb, _m_gain = dca_bootstrap(dca_df, dca_thresholds, n_boot=2000)

dca_curve["nb_model_lo"] = np.percentile(_m_nb, 2.5, axis=0)
dca_curve["nb_model_hi"] = np.percentile(_m_nb, 97.5, axis=0)
dca_curve["gain_lo"]     = np.percentile(_m_gain, 2.5, axis=0)
dca_curve["gain_hi"]     = np.percentile(_m_gain, 97.5, axis=0)
dca_curve["prob_gain_positive"] = (_m_gain > 0).mean(axis=0)

print(dca_curve[dca_curve["threshold"].isin(_show)][
    ["threshold", "nb_model", "nb_model_lo", "nb_model_hi",
     "nb_gain_vs_best_default", "gain_lo", "gain_hi",
     "prob_gain_positive"]].round(4).to_string(index=False))

In [ ]:
DCA_ORANGE = "#d95f02"
DCA_BLUE = "#2b6cb0"
DCA_BAND = "#fde68a"
DCA_GRAY = "#6b7280"
DCA_INK = "#111827"

### Standardized decision curve analysis

In [ ]:
def _dca_nb_fast(y, p, th):
    n, prev = len(y), y.mean()
    p1, p0 = np.sort(p[y == 1]), np.sort(p[y == 0])
    tp = (len(p1) - np.searchsorted(p1, th, side="left")) / n
    fp = (len(p0) - np.searchsorted(p0, th, side="left")) / n
    w = th / (1 - th)
    return tp - fp * w, prev - (1 - prev) * w, prev      # <- now also returns prevalence

def dca_bootstrap_snb(df, th, n_boot=2000, seed=42):
    """As before, but each draw is standardised by ITS OWN prevalence."""
    rng = np.random.default_rng(seed)
    _g = {pid: (s["y"].to_numpy(int), s["p"].to_numpy(float))
          for pid, s in df.groupby("patient_id")}
    _pids = np.array(list(_g.keys()))
    m_nb, m_gain  = np.empty((n_boot, len(th))), np.empty((n_boot, len(th)))
    m_snb, m_sgain = np.empty((n_boot, len(th))), np.empty((n_boot, len(th)))
    for b in range(n_boot):
        s = rng.choice(_pids, size=len(_pids), replace=True)
        _yb = np.concatenate([_g[i][0] for i in s])
        _pb = np.concatenate([_g[i][1] for i in s])
        nb, na, prev_b = _dca_nb_fast(_yb, _pb, th)
        gain = nb - np.maximum(na, 0.0)
        m_nb[b], m_gain[b] = nb, gain
        if prev_b > 0:
            m_snb[b], m_sgain[b] = nb / prev_b, gain / prev_b
        else:
            m_snb[b] = m_sgain[b] = np.nan
    return m_nb, m_gain, m_snb, m_sgain

_m_nb, _m_gain, _m_snb, _m_sgain = dca_bootstrap_snb(dca_df, dca_thresholds, n_boot=2000)

dca_prev = dca_df["y"].mean()
dca_curve["snb_model"]      = dca_curve["nb_model"]     / dca_prev
dca_curve["snb_treat_all"]  = dca_curve["nb_treat_all"] / dca_prev
dca_curve["snb_treat_none"] = 0.0
dca_curve["snb_gain_vs_best_default"] = dca_curve["nb_gain_vs_best_default"] / dca_prev

dca_curve["nb_model_lo"]  = np.percentile(_m_nb, 2.5, axis=0)
dca_curve["nb_model_hi"]  = np.percentile(_m_nb, 97.5, axis=0)
dca_curve["gain_lo"]      = np.percentile(_m_gain, 2.5, axis=0)
dca_curve["gain_hi"]      = np.percentile(_m_gain, 97.5, axis=0)
dca_curve["snb_model_lo"] = np.nanpercentile(_m_snb, 2.5, axis=0)
dca_curve["snb_model_hi"] = np.nanpercentile(_m_snb, 97.5, axis=0)
dca_curve["snb_gain_lo"]  = np.nanpercentile(_m_sgain, 2.5, axis=0)
dca_curve["snb_gain_hi"]  = np.nanpercentile(_m_sgain, 97.5, axis=0)
dca_curve["prob_gain_positive"] = (_m_gain > 0).mean(axis=0)

print(f"prevalence = {dca_prev:.4f}  →  max attainable net benefit = {dca_prev:.4f}, sNB caps at 1.0\n")
print(dca_curve[dca_curve["threshold"].isin(_show)][
    ["threshold", "nb_model", "snb_model", "snb_model_lo",
     "snb_model_hi", "snb_treat_all"]].round(3).to_string(index=False))

In [ ]:
def _cb_label(x, maxden=100):
    f = Fraction(x / (1 - x)).limit_denominator(maxden)
    return f"{f.numerator}:{f.denominator}"


def dca_plot_snb(curve, label="", ref_threshold=0.20, show_reduction=False,
                 standardized=True, xlim=(0, 1), aspect=0.62,
                 plot_data=False, plot_data_name=None):     # <-- NEW
    key  = "snb_model"     if standardized else "nb_model"
    kall = "snb_treat_all" if standardized else "nb_treat_all"
    klo, khi = ("snb_model_lo", "snb_model_hi") if standardized else ("nb_model_lo", "nb_model_hi")
    ylab = "Standardized net benefit" if standardized else "Net benefit"

    n = 2 if show_reduction else 1
    fig, axes = plt.subplots(n, 1, figsize=(6.4, 7.0) if n == 2 else (6.2, 6.2), sharex=True,
                             gridspec_kw={"height_ratios": [2, 1][:n]})
    axes = np.atleast_1d(axes); ax = axes[0]
    t = curve["threshold"].to_numpy()

    #_thr = ax.axvline(ref_threshold, color=DCA_ORANGE, lw=1.6, zorder=2)
    if klo in curve:
        ax.fill_between(t, curve[klo], curve[khi], color=DCA_BLUE, alpha=.18, lw=0, zorder=1)
    ax.plot(t, curve[kall], color=DCA_GRAY, lw=2, ls="--", zorder=2, label="Treat all")
    ax.axhline(0, color=DCA_INK, lw=1.6, ls=":", zorder=2, label="Treat none")
    ax.plot(t, curve[key], color=DCA_BLUE, lw=2.4, zorder=3, label="Model")

    if standardized:
        #ax.axhline(1.0, color=DCA_INK, lw=0.8, alpha=.35, zorder=1)
        #ax.text(t.min(), 1.008, "maximum attainable", ha="left", va="bottom",
        #        fontsize=7.5, color=DCA_INK, alpha=.65)
        ax.set_ylim(-0.60, 1.06)
    else:
        ax.set_ylim(max(min(0, curve[key].min()) - 0.01, -0.05), curve[key].max() * 1.18)

    ax.set_ylabel(ylab)
    ax.set_title(f"Standardized decision curve analysis{' — ' + label if label else ''}", loc="left", pad=26)
    _h, _l = ax.get_legend_handles_labels()
    #_h.append(_thr); _l.append("Model decision threshold")
    ax.legend(_h, _l, frameon=False, fontsize=9, loc="upper right")
    ax.grid(axis="y", color="k", alpha=.06, lw=.8); ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

    _win = curve.loc[curve["nb_gain_vs_best_default"] > 0, "threshold"]

    sec = ax.secondary_xaxis("top")
    _tk = [x for x in (1/101, 0.20, 0.40, 0.60, 0.80, 100/101) if xlim[0] <= x <= xlim[1]]
    sec.set_xticks(_tk)
    sec.set_xticklabels([_cb_label(x) for x in _tk], fontsize=8)
    sec.set_xlabel("Cost : benefit ratio", fontsize=8.5, labelpad=4, color=DCA_INK)
    sec.spines["top"].set_visible(False); sec.tick_params(length=3, colors=DCA_INK)

    if show_reduction:
        ax2 = axes[1]
        #ax2.axvline(ref_threshold, color=DCA_ORANGE, lw=1.6, zorder=2)
        ax2.axhline(0, color=DCA_INK, lw=1, ls=":", zorder=2)
        ax2.plot(t, curve["net_reduction_per_100"], color=DCA_BLUE, lw=2.4, zorder=3)
        ax2.set_ylabel("Antibiotic starts avoided\nper 100 patient-days\n(vs treat all)")
        ax2.grid(axis="y", color="k", alpha=.06, lw=.8); ax2.set_axisbelow(True)
        ax2.spines[["top", "right"]].set_visible(False)

    axes[-1].set_xlabel("Decision threshold")
    for _a in axes:
        _a.set_xlim(*xlim)

    ax.set_anchor("S"); axes[-1].set_anchor("N")
    ax.set_box_aspect(aspect)
    if n == 2:
        axes[1].set_box_aspect(aspect / 2)
    fig.tight_layout()

    # ---- source data for the journal --------------------------------------
    if plot_data:
        import pandas as pd
        pfx = "snb" if standardized else "nb"
        with np.errstate(divide="ignore", invalid="ignore"):
            _cb = np.where(t < 1, t / (1 - t), np.nan)
        _cb = np.where(np.isfinite(_cb), _cb, np.nan)

        nb = {"threshold": t, "cost_benefit_ratio": _cb,
              f"{pfx}_model": curve[key].to_numpy()}
        if klo in curve:
            nb[f"{pfx}_model_ci_lower"] = curve[klo].to_numpy()
            nb[f"{pfx}_model_ci_upper"] = curve[khi].to_numpy()
        nb[f"{pfx}_treat_all"] = curve[kall].to_numpy()
        nb[f"{pfx}_treat_none"] = np.zeros_like(t, dtype=float)
        tables = {"net_benefit": pd.DataFrame(nb)}

        if show_reduction:
            tables["net_reduction"] = pd.DataFrame({
                "threshold": t,
                "net_reduction_per_100": curve["net_reduction_per_100"].to_numpy(),
            })

        write_plot_data(
            plot_data_name or f"{pfx}_decision_curve",
            tables,
            out_dir=PLOT_DATA_PATH,
            note=(f"{ylab}; {'with' if klo in curve else 'no'} 95% CI band; "
                  f"{len(t)} thresholds ({t.min():.4g}-{t.max():.4g}); "
                  f"plotted x-range {xlim[0]}-{xlim[1]}"
                  + (f"; cohort: {label}" if label else "")),
        )

    return fig


DCA_NAME = 'DCA_standardized'
fig = dca_plot_snb(dca_curve, label="Developement cohort", ref_threshold=0.20,
                   plot_data=True, plot_data_name=DCA_NAME)
fig.savefig(PLOT_PATH+'simple_model/'+DCA_NAME, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
_show = [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
print(f"prevalence = {dca_prev:.4f}\n")
print(dca_curve[dca_curve["threshold"].isin(_show)][
    ["threshold", "snb_model", "snb_model_lo", "snb_model_hi",
     "snb_treat_all", "snb_gain_lo", "snb_gain_hi"]].round(4).to_string(index=False))

### Linear regression to event days

#### Parameters

In [247]:
EVENT_TO_HOURS = {1: -48, 2: -24, 3: 0}
FN_THRESHOLD   = 38.0
PATIENT_COL    = "henkilotunnus"
CYCLE_COL      = "cycle_number"

# ordinal blue ramp: light = furthest from FN, dark = FN day
COLORS = {-48: "#6BAED6", -24: "#2171B5", 0: "#08306B"}
LABELS = {-48: "48 h before FN", -24: "24 h before FN", 0: "FN day"}
INK, MUTED, ACCENT = "#1c1c1c", "#8a8a8a", "#ED101B"

plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.titlesize": 11})

#### Data

In [248]:
trdf = pd.read_csv(DATA_PATH + 'df_regression_tr.csv')
testdf = pd.read_csv(DATA_PATH + 'df_regression_test.csv')

In [ ]:
d = trdf[trdf["event"].isin(EVENT_TO_HOURS)].copy()
d["hours"]    = d["event"].map(EVENT_TO_HOURS)
d["is_fn"]    = (d["hours"] == 0).astype(int)
d["cycle_id"] = d[PATIENT_COL].astype(str) + "__c" + d[CYCLE_COL].astype(str)
d = d.dropna(subset=["temperature"])

g = d.groupby("hours")["temperature"]
desc = pd.DataFrame({
    "n": g.size(), "mean": g.mean(), "sd": g.std(), "median": g.median(),
    "pct_ge_38": d.assign(x=d["temperature"] >= FN_THRESHOLD)
                  .groupby("hours")["x"].mean() * 100,
})
desc["se"] = desc["sd"] / np.sqrt(desc["n"])
desc.round(3)

#### Regressions

In [ ]:
# A: fitted on all three timepoints — the circular one
m_all = smf.mixedlm("temperature ~ hours", d, groups=d[PATIENT_COL]).fit(reml=False)

# B: fitted on −48 h and −24 h only — never sees the FN day
pre   = d[d["hours"] < 0]
m_pre = smf.mixedlm("temperature ~ hours", pre, groups=pre[PATIENT_COL]).fit(reml=False)

# C: saturated — the is_fn coefficient IS the excess over B's extrapolation
m_jump = smf.mixedlm("temperature ~ hours + is_fn", d, groups=d[PATIENT_COL]).fit(reml=False)

print(f"A  all three timepoints : {m_all.params['hours']*24:+.3f} °C/day")
print(f"B  −48/−24 h only       : {m_pre.params['hours']*24:+.3f} °C/day")

b0 = m_pre.params["Intercept"]                       # = B's prediction at 0 h
V  = m_pre.cov_params().loc[["Intercept","hours"], ["Intercept","hours"]].values
se0 = np.sqrt(np.array([1,0]) @ V @ np.array([1,0]))
print(f"\nB extrapolated to 0 h : {b0:.2f} °C (95% CI {b0-1.96*se0:.2f}–{b0+1.96*se0:.2f})")
print(f"observed at 0 h       : {desc.loc[0,'mean']:.2f} °C")

exc, exc_se = m_jump.params["is_fn"], m_jump.bse["is_fn"]
print(f"\nExcess at 0 h: {exc:+.2f} °C "
      f"(95% CI {exc-1.96*exc_se:+.2f} to {exc+1.96*exc_se:+.2f}), "
      f"p = {m_jump.pvalues['is_fn']:.3g}")

lr = 2 * (m_jump.llf - m_all.llf)
print(f"Departure from linearity: LR χ²(1) = {lr:.1f}, p = {stats.chi2.sf(lr,1):.3g}")

In [ ]:
pre = d[d["hours"] < 0]

# cycles contributing each pre-onset timepoint
wide_pre = (pre.pivot_table(index="cycle_id", columns="hours",
                            values="temperature", aggfunc="mean")
              .rename(columns={-48: "t48", -24: "t24"}))
complete = wide_pre.dropna(subset=["t48", "t24"]).index

print(f"cycles with −48 h: {pre[pre.hours == -48].cycle_id.nunique()} | "
      f"with −24 h: {pre[pre.hours == -24].cycle_id.nunique()} | "
      f"with both: {len(complete)}")

# --- models, now grouped by cycle rather than patient -------------------
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m_all  = smf.mixedlm("temperature ~ hours",         d,   groups=d["cycle_id"]).fit(reml=False)
    m_pre  = smf.mixedlm("temperature ~ hours",         pre, groups=pre["cycle_id"]).fit(reml=False)
    m_jump = smf.mixedlm("temperature ~ hours + is_fn", d,   groups=d["cycle_id"]).fit(reml=False)

for name, m in [("m_all", m_all), ("m_pre", m_pre), ("m_jump", m_jump)]:
    if not m.converged:
        print(f"  !! {name} did not converge — check before using")

print(f"\nA  all three timepoints : {m_all.params['hours']*24:+.3f} °C/day")
print(f"B  −48/−24 h only       : {m_pre.params['hours']*24:+.3f} °C/day "
      f"(p = {m_pre.pvalues['hours']:.3g})")

b0  = m_pre.params["Intercept"]                       # B's prediction at 0 h
V   = m_pre.cov_params().loc[["Intercept","hours"], ["Intercept","hours"]].values
se0 = np.sqrt(np.array([1,0]) @ V @ np.array([1,0]))
print(f"\nB extrapolated to 0 h : {b0:.2f} °C (95% CI {b0-1.96*se0:.2f}–{b0+1.96*se0:.2f})")
print(f"observed at 0 h       : {desc.loc[0,'mean']:.2f} °C")

exc, exc_se = m_jump.params["is_fn"], m_jump.bse["is_fn"]
print(f"\nExcess at 0 h: {exc:+.2f} °C (95% CI {exc-1.96*exc_se:+.2f} to "
      f"{exc+1.96*exc_se:+.2f}), p = {m_jump.pvalues['is_fn']:.3g}")

lr = 2 * (m_jump.llf - m_all.llf)
print(f"Departure from linearity: LR χ²(1) = {lr:.1f}, p = {stats.chi2.sf(lr, 1):.3g}")

# --- pure within-cycle change, no composition at all --------------------
dlt  = wide_pre.loc[complete, "t24"] - wide_pre.loc[complete, "t48"]
nn   = len(dlt)
half = stats.t.ppf(.975, nn-1) * dlt.std(ddof=1) / np.sqrt(nn)
print(f"\nPaired within-cycle change −48→−24 h (n = {nn}): {dlt.mean():+.3f} °C "
      f"(95% CI {dlt.mean()-half:+.3f} to {dlt.mean()+half:+.3f}), "
      f"p = {stats.ttest_1samp(dlt, 0).pvalue:.3g}")

# --- conservative flat-baseline comparison ------------------------------
flat = d.loc[d.hours < 0, "temperature"].mean()
print(f"Flat-baseline comparison: pooled pre-onset mean {flat:.2f} °C, "
      f"difference to 0 h {desc.loc[0,'mean']-flat:+.2f} °C")

In [ ]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(6.8, 5.0), dpi=150)

# observations, coloured by event
for h in (-48, -24, 0):
    s = d[d["hours"] == h]

# distribution at each timepoint: box + every observation as a jittered dot
_hs  = (-48, -24, 0)
_grp = [d.loc[d["hours"] == h, "temperature"].to_numpy() for h in _hs]
bp = ax.boxplot(_grp, positions=list(_hs), widths=9, patch_artist=True,
                showfliers=False, manage_ticks=False, zorder=3,
                medianprops=dict(color=INK, lw=2.2),
                boxprops=dict(edgecolor=INK, lw=1.2),
                whiskerprops=dict(color=INK, lw=1.2),
                capprops=dict(lw=0))
for patch, h in zip(bp["boxes"], _hs):
    patch.set_facecolor(mcolors.to_rgba(COLORS[h], 0.60))
    patch.set_label(LABELS[h])

for h in _hs:
    s = d[d["hours"] == h]
    ax.scatter(s["hours"] + rng.uniform(-3.2, 3.2, len(s)), s["temperature"],
               s=12, color=INK, alpha=.25, lw=0, zorder=4)
    # ----------------------

# the circular fit — shown in order to be rejected
xl = np.array([-48, 0])
ax.plot(xl, m_all.params["Intercept"] + m_all.params["hours"]*xl,
        color=INK, lw=1.6, alpha=.55, zorder=4, label="Observed temperature regression curve");

# the honest fit: −48/−24 h only, extrapolated to 0 h, with 95% CI band
xs   = np.linspace(-48, 4, 60)
yhat = m_pre.params["Intercept"] + m_pre.params["hours"]*xs
Xd   = np.column_stack([np.ones_like(xs), xs])
seb  = np.sqrt(np.einsum("ij,jk,ik->i", Xd, V, Xd))
ax.fill_between(xs, yhat-1.96*seb, yhat+1.96*seb, color=ACCENT, alpha=.15, zorder=2);
ax.plot(xs[xs <= -24], yhat[xs <= -24], color=ACCENT, lw=2, zorder=4);
ax.plot(xs[xs >= -24], yhat[xs >= -24], color=ACCENT, lw=2, ls=(0,(4,3)), zorder=4,
        label="Prediction-window regression curve extrapolated to the FN day");

print(f"{exc:+.2f} °C\nunexplained")

ax.axhline(FN_THRESHOLD, color=MUTED, lw=1, ls=":", zorder=0)

ax.set_xticks([-48, -24, 0]); ax.set_xticklabels(["−48 h", "−24 h", "FN day"])
ax.set_xlim(-56, 20)
ax.set_xlabel("Time relative to FN onset")
ax.set_ylabel("Body temperature (°C)")
ax.grid(axis="y", color="#e6e6e6", lw=.8); ax.set_axisbelow(True)
for sd in ("top", "right"):   ax.spines[sd].set_visible(False)
for sd in ("left", "bottom"): ax.spines[sd].set_color("#cccccc")
ax.legend(frameon=False, fontsize=8, loc="upper left")

fig.tight_layout()
fig.savefig(PLOT_PATH+"simple_model/fig_temperature_extrapolation.png", dpi=300, bbox_inches="tight");
#fig.savefig("fig_temperature_extrapolation.pdf", bbox_inches="tight")

# ---- source data for the journal ----------------------------------------
FIG_NAME = "fig_temperature_extrapolation"
PLOT_DATA_POINTS = True          # set False to leave the raw dots out

_st = cbook.boxplot_stats(_grp, whis=1.5)
boxes = pd.DataFrame({
    "hours":        list(_hs),
    "timepoint":    [LABELS[h] for h in _hs],
    "n":            [int(len(v)) for v in _grp],
    "whisker_low":  [s["whislo"] for s in _st],
    "q1":           [s["q1"] for s in _st],
    "median":       [s["med"] for s in _st],
    "q3":           [s["q3"] for s in _st],
    "whisker_high": [s["whishi"] for s in _st],
    "mean":         [s["mean"] for s in _st],
})

fits = pd.concat([
    pd.DataFrame({
        "curve": "fitted on prediction window + FN day",
        "hours": xl.astype(float),
        "temperature_fitted": m_all.params["Intercept"] + m_all.params["hours"]*xl,
        "ci_lower": np.nan, "ci_upper": np.nan,
        "line_style": "solid",
    }),
    pd.DataFrame({
        "curve": "fitted on prediction window only",
        "hours": xs,
        "temperature_fitted": yhat,
        "ci_lower": yhat - 1.96*seb,
        "ci_upper": yhat + 1.96*seb,
        "line_style": np.where(xs <= -24, "solid", "dashed"),
    }),
], ignore_index=True)

summary = pd.DataFrame([{
    "fn_threshold_c":         float(FN_THRESHOLD),
    "unexplained_excess_c":   float(exc),
    "intercept_all_data":     float(m_all.params["Intercept"]),
    "slope_all_data_per_h":   float(m_all.params["hours"]),
    "intercept_pre_window":   float(m_pre.params["Intercept"]),
    "slope_pre_window_per_h": float(m_pre.params["hours"]),
    "n_observations":         int(len(d)),
}])

tables = {"boxplots": boxes, "fitted_curves": fits, "summary": summary}
if PLOT_DATA_POINTS:
    tables["observations"] = pd.DataFrame({
        "hours":       d["hours"].to_numpy(),
        "timepoint":   [LABELS[h] for h in d["hours"]],
        "temperature": d["temperature"].to_numpy(),
    })

write_plot_data(
    FIG_NAME, tables, out_dir=PLOT_DATA_PATH,
    note=(f"boxes = median/IQR, whiskers 1.5*IQR, fliers not drawn; "
          f"dots jittered +/-3.2 h for display only (x here is nominal); "
          f"band = 95% CI of the prediction-window fit; "
          f"unexplained excess {exc:+.2f} C; FN threshold {FN_THRESHOLD} C"),
)

In [ ]:
print(f"n = {int(desc['n'].sum())} observations from {d['cycle_id'].nunique()} cycles "
      f"in {d[PATIENT_COL].nunique()} patients")
print(f"slope −48→−24 h : {m_pre.params['hours']*24:+.3f} °C/day "
      f"(p = {m_pre.pvalues['hours']:.3g})")
print(f"predicted 0 h   : {b0:.2f} °C   observed 0 h: {desc.loc[0,'mean']:.2f} °C")
print(f"excess          : {exc:+.2f} °C (95% CI {exc-1.96*exc_se:+.2f} to "
      f"{exc+1.96*exc_se:+.2f}), p = {m_jump.pvalues['is_fn']:.3g}")

### Individual ID visualisations

In [310]:

sigma = 1.2  # ~“radius” in samples; larger = smoother

def plot_random_id_cycles_scatter_with_true_labels(
    df,
    id_column="henkilotunnus",
    cycle_column="cycle_number",
    proba_column="pred_proba",
    true_column="original_y",
    random_state=None,
    save_path=None,
    # labs present in the dataframe (machine names)
    lab_cols=("temperature", "b_neut", "p_crp"),
    # pretty display names for legend/ylabels – same order as lab_cols
    colnames=("Body temperature", "ANC", "CRP"),
    # y-limits and tick steps per right axis – same order as lab_cols
    ranges=((35, 40), (0, 5), (0, 100)),
    ticks=(0.5, 0.5, 10),
    # units shown next to ylabel
    lab_units=None,
    random_id=None,
    label=None,
    THRESHOLD=THRESHOLD,
    plot_data=False
):
    """
    Draw one cycle per figure for a random id (or the provided id).
    Left y-axis: prediction probability scatter (0–1).
    Right y-axes: lab time series, each on its own axis.

    The legend names for lab lines come from `colnames`.
    """
    if lab_units is None:
        lab_units = {"temperature": "°C", "b_neut": "10^9/L", "p_crp": "mg/L"}

    rng = np.random.default_rng(random_state)

    if random_id is None:
        random_id = rng.choice(df[id_column].unique())
        print(random_id)

    subset = df[df[id_column] == random_id].reset_index(drop=True)

    for cycle in subset[cycle_column].unique():
        g = subset[subset[cycle_column] == cycle].reset_index(drop=True)

        # --- X positions (your convention)
        # x_vals = np.arange(5, 5 + len(g))
        x_vals = g["countdown"].to_numpy()+1 # starts from 1 instead of 0
        y_pred = g[proba_column].to_numpy()
        #colors = g[true_column].map({0: "tab:blue", 1: "tab:red"}).to_numpy()
        colors = g[true_column].map({0: "tab:blue", 1: "tab:red"}).fillna("tab:red").to_numpy()

        ind_type = (
            "Induction"
            if "sykli_IND" in g.columns and g["sykli_IND"].iloc[0] == 1
            else "Consolidation"
        )

        fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=False)
        ax.scatter(x_vals, y_pred, c=colors, alpha=0.8, s=30)
        ax.set_xticks(x_vals)
        ax.set_xlabel("Days", fontsize=15)
        ax.set_ylabel("Predicted FN risk", fontsize=15)
        
        if label is None:
            ax.set_title(f"Cycle {cycle} ({ind_type})", fontsize=15, pad=49)
        else:
            ax.set_title(label, fontsize=16, pad=49)

        # Optional classification threshold line if present in scope
        try:
            ax.axhline(
                y=THRESHOLD,
                linestyle="--",
                alpha=0.9,
                linewidth=1.5,
                label=f"Decision threshold",
                c="grey",
            )
        except NameError:
            pass

        # --- labs on right axes -------------------------------------------------
        right_axes = []
        valid_labs = [c for c in lab_cols if c in g.columns]
        _smoothed = {}

        # keep only matching display names / limits / steps by index of valid_labs
        # (assumes relative order is preserved)
        colnames_used = [colnames[lab_cols.index(l)] for l in valid_labs]
        ranges_used = [ranges[lab_cols.index(l)] for l in valid_labs]
        ticks_used = [ticks[lab_cols.index(l)] for l in valid_labs]

        for i, lab in enumerate(valid_labs):
            series = g[lab].astype(float).ffill()  # forward-fill within cycle

            # --- Gaussian smooth (assumes roughly uniform x spacing)
            y_smooth = gaussian_filter1d(series.to_numpy(), sigma=sigma, mode="nearest")
            #y_smooth = series.to_numpy()
            _smoothed[lab] = (series.to_numpy(), y_smooth)

            ax_r = ax.twinx()
            ax_r.set_zorder(0)
            ax_r.spines["right"].set_position(("axes", 1 + 0.10 * i))
            ax_r.set_frame_on(False)
            ax_r.patch.set_visible(False)
            for s in ("left", "top", "bottom"):
                ax_r.spines[s].set_visible(False)

            # --- use pretty name for label & ylabel
            display_name = colnames_used[i]
            unit = f" ({lab_units.get(lab, '')})" if lab in lab_units else ""

            ax_r.plot(
                x_vals,
                #series.to_numpy(),
                y_smooth,
                linestyle="-",
                label=display_name,         # << legend text from colnames
                linewidth=2,
                # color=f"tab:{['green','orange','purple'][i%6]}",
                color=f"{['#009E73','#E69F00','#CC79A7'][i%6]}",
                alpha=0.5,
            )
            ax_r.set_ylabel(f"{display_name}{unit}", rotation=270, labelpad=13, fontsize=13)
            ax_r.set_ylim(ranges_used[i])
            ax_r.tick_params(axis='y', which='major', labelsize=13)
            ax_r.yaxis.set_major_locator(MultipleLocator(ticks_used[i]))

            right_axes.append(ax_r)

        # --- build legend -------------------------------------------------------
        # infection color patches (to explain scatter colors)
        blue_patch = mpatches.Patch(color="#4A7FBF", label="Non infection")
        red_patch = mpatches.Patch(color="#D95F02", label="Infection")

        # left-axis items (e.g., threshold line)
        left_h, left_l = ax.get_legend_handles_labels()

        # right-axis lab lines (already labelled with colnames)
        lab_h, lab_l = [], []
        for ax_r in right_axes:
            h, l = ax_r.get_legend_handles_labels()
            lab_h += h
            lab_l += l

        handles = [blue_patch, red_patch] + left_h + lab_h
        labels = ["Non-FN", "FN"] + left_l + lab_l

        ax.legend(
            handles=handles,
            labels=labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.23),
            ncol=3,
            fontsize=13,
            frameon=True,
        )

        # cosmetics for left axis
        ax.grid(True, alpha=0.2)
        ax.set_ylim(0, 1)
        ax.tick_params(axis='y', which='major', labelsize=13)
        ax.tick_params(axis='x', which='major', labelsize=13)
        ax.yaxis.set_major_locator(MultipleLocator(0.1))

        if save_path is not None:
            plt.savefig(f"{save_path}/{random_id}_cycle_{cycle}_{label}",
                        dpi=300, bbox_inches="tight")

        # ---- source data for the journal -----------------------------------
        if plot_data:
            _tab = pd.DataFrame({
                "day":               x_vals,
                "countdown":         g["countdown"].to_numpy(),
                "predicted_fn_risk": y_pred,
                "true_label":        g[true_column].to_numpy(),
                "point_class":       g[true_column].map({0: "Non-FN", 1: "FN"}).to_numpy(),
            })
            for lab in valid_labs:
                _raw, _sm = _smoothed[lab]
                _tab[lab] = _raw
                _tab[f"{lab}_smoothed"] = _sm
            write_plot_data(
                f"{random_id}_cycle_{cycle}_{label}", _tab,
                out_dir=PLOT_DATA_PATH,
                note=(f"patient={random_id}; cycle={cycle}; {ind_type}; "
                      f"panel label={label}; decision threshold={THRESHOLD}; "
                      f"right axes draw gaussian_filter1d(sigma={sigma}, mode=nearest) "
                      f"of the forward-filled series -- '*_smoothed' is what is drawn; "
                      f"display names: "
                      + ", ".join(f"{l}={colnames_used[i]}"
                                  for i, l in enumerate(valid_labs))),
            )

        plt.show()

    return random_id


In [ ]:
random_id0 = plot_random_id_cycles_scatter_with_true_labels(eval_df, random_id='02139_42184194', label="Correct prediction", save_path=PLOT_PATH+'simple_model/individual_plots', plot_data=False)
random_id0 = plot_random_id_cycles_scatter_with_true_labels(eval_df, random_id='02139_42184194', label="False-negative prediction", save_path=PLOT_PATH+'simple_model/individual_plots', plot_data=True)
random_id1 = plot_random_id_cycles_scatter_with_true_labels(eval_df, random_id='02139_65047479', label="False-positive prediction", save_path=PLOT_PATH+'simple_model/individual_plots', plot_data=True)
random_id2 = plot_random_id_cycles_scatter_with_true_labels(eval_df, random_id='02139_85874893', label="Early prediction", save_path=PLOT_PATH+'simple_model/individual_plots', plot_data=True)

### Induction and consolidation

In [ ]:
# we can use earlier created eval_df
ind = eval_df.loc[eval_df['sykli_IND']==1]

#df, score, y_pred_mod = custom_score(ind['y_true'], y_prob_test, group_test, threshold=0.29, eval=True)
y_test = ind['infektion_binary']
y_pred_mod = ind['y_pred']
y_pred_ = (ind['pred_proba'] >= THRESHOLD).astype(int)

# 👉 3. Evaluation metrics
f1 = f1_score(y_test, y_pred_mod)
recall = recall_score(y_test, y_pred_mod)
accuracy = accuracy_score(y_test, y_pred_mod)
precision = precision_score(y_test, y_pred_mod)
kappa = cohen_kappa_score(y_test, y_pred_mod)

print(f"Kappa       : {kappa:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
customf1 = custom_f1(y_test, y_pred_mod)
print(f"Custom F1   : {customf1:.4f}")

# build a metrics table
rows = pd.DataFrame({
    'data':['simple_IND', 'simple_IND_corrected'],
    'F1_score': [f1_score(y_test, y_pred_), f1_score(y_test, y_pred_mod)],
    'kappa' : [cohen_kappa_score(y_test, y_pred_), cohen_kappa_score(y_test, y_pred_mod)],
    'recall': [recall_score(y_test, y_pred_), recall_score(y_test, y_pred_mod)],
    'precision' : [precision_score(y_test, y_pred_), precision_score(y_test, y_pred_mod)],
    'accuracy' : [accuracy_score(y_test, y_pred_), accuracy_score(y_test, y_pred_mod)],
    'customf1' : [custom_f1(y_test, y_pred_), custom_f1(y_test, y_pred_mod)],
    'specificity':[recall_score(y_test, y_pred_, pos_label=0), recall_score(y_test, y_pred_mod, pos_label=0)],
    'PNV':[calc_pnv(y_test, y_pred_), calc_pnv(y_test, y_pred_mod)]

})
metrics_df = pd.concat([metrics_df, rows], ignore_index=True)

plot_confusion_matrix_with_percentages(
    y_test, y_pred_mod, 
    percent_type='row', 
    threshold=THRESHOLD,
    #title='Induction: Confusion Matrix\nTest set',
    # title='Test set, Inductions\nConfusion matrix with row %',
    title='Test set, Inductions',
    save_path=PLOT_PATH+'simple_model/cm_induction_'+str(THRESHOLD).replace('.', '_'),
    cmap=colourmap,
    text_color='white',
    plot_data=True
)

display(metrics_df.style.format({"Value": "{:.4f}"}))

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_,  THRESHOLD, "ind_top")
metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "ind_top_corrected")
fmt_table(metrics_df_ci)

In [ ]:
kons = eval_df.loc[eval_df['sykli_IND']!=1]


#df, score, y_pred_mod = custom_score(kons['y_true'], y_prob_test, group_test, threshold=0.29, eval=True)
y_test = kons['infektion_binary']
y_pred_mod = kons['y_pred']
y_pred_ = (kons['pred_proba'] >= THRESHOLD).astype(int)

# 👉 3. Evaluation metrics
f1 = f1_score(y_test, y_pred_mod)
recall = recall_score(y_test, y_pred_mod)
accuracy = accuracy_score(y_test, y_pred_mod)
precision = precision_score(y_test, y_pred_mod)
kappa = cohen_kappa_score(y_test, y_pred_mod)

print(f"Kappa       : {kappa:.4f}")
print(f"F1 Score    : {f1:.4f}")
print(f"Recall      : {recall:.4f}")
print(f"Accuracy    : {accuracy:.4f}")
print(f"Precision   : {precision:.4f}")
customf1 = custom_f1(y_test, y_pred_mod)
print(f"Custom F1   : {customf1:.4f}")

# build a metrics table
rows = pd.DataFrame({
    'data':['simple_KONS', 'simple_KONS_corrected'],
    'F1_score': [f1_score(y_test, y_pred_), f1_score(y_test, y_pred_mod)],
    'kappa' : [cohen_kappa_score(y_test, y_pred_), cohen_kappa_score(y_test, y_pred_mod)],
    'recall': [recall_score(y_test, y_pred_), recall_score(y_test, y_pred_mod)],
    'precision' : [precision_score(y_test, y_pred_), precision_score(y_test, y_pred_mod)],
    'accuracy' : [accuracy_score(y_test, y_pred_), accuracy_score(y_test, y_pred_mod)],
    'customf1' : [custom_f1(y_test, y_pred_), custom_f1(y_test, y_pred_mod)],
    'specificity':[recall_score(y_test, y_pred_, pos_label=0), recall_score(y_test, y_pred_mod, pos_label=0)],
    'PNV':[calc_pnv(y_test, y_pred_), calc_pnv(y_test, y_pred_mod)]


})
metrics_df = pd.concat([metrics_df, rows], ignore_index=True)
#metrics_df

plot_confusion_matrix_with_percentages(
    y_test, y_pred_mod, 
    percent_type='row', 
    threshold=THRESHOLD,
    title='Test set, Consolidations',
    save_path=PLOT_PATH+'simple_model/cm_consolidation_'+str(THRESHOLD).replace('.', '_'),
    cmap=colourmap,
    text_color='white',
    plot_data=True
)

display(metrics_df.style.format({"Value": "{:.4f}"}))

metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_,  THRESHOLD, "kons_top")
metrics_df_ci = add_metrics_column(metrics_df_ci, y_test, y_pred_mod,  THRESHOLD, "kons_top_corrected")


### Performance summary table (both models)

In [274]:
metrics_df.to_csv(PLOT_PATH+'metrics_table_all.csv')
metrics_df_ci.to_csv(PLOT_PATH+'CI_metrics_table_TOP.csv')

In [279]:
# Misclassified vs correctly classified by feature
eval_df['misclassified'] = (eval_df['pred_modified_y'] != eval_df['original_y']).astype(int)
eval_df[eval_df['misclassified']==1].to_csv(DATA_PATH+'misclassified_rows')